# 08_integration_check

Notebook này dùng để **kiểm tra end-to-end ở mức inference / integration**, bám trực tiếp vào các artifact đã được tạo từ:

- `02_data_preprocessing_master_dataset.ipynb`
- `03_classification_pipeline.ipynb`
- `04_regression_pipeline.ipynb`
- `05_clustering_rfm.ipynb`
- `06_surprise_recommendation.ipynb`
- `07_fpgrowth_rules.ipynb`

## Vai trò của file 08 trong toàn dự án
File 08 **không train lại mô hình** và **không sinh feature mới**.  
Nó là lớp **kiểm tra tích hợp cuối** để trả lời 4 câu hỏi:

1. Các **processed table / model / summary / prediction artifact** có tồn tại không?
2. Các model/bundle có **load lại được** không?
3. Input inference cho từng nhánh có **đúng schema tối thiểu** không?
4. Có thể tạo **payload sẵn sàng cho Streamlit / UI** cho:
   - Classification
   - Regression
   - Clustering
   - Recommendation
   - FP-Growth

## Phạm vi phụ thuộc
- **Đọc** output từ notebook `02` đến `07`
- **Không sửa** notebook upstream
- **Không thay đổi** tên file artifact lõi
- **Chỉ ghi thêm** các file audit / summary riêng cho integration

## Tiêu chí PASS của notebook này
Notebook chỉ được xem là **PASS ở mức core** khi:
- các file artifact cốt lõi tồn tại,
- model/bundle quan trọng load được,
- smoke test suy luận cho các nhánh chính không lỗi,
- payload UI-ready được tạo đủ,
- `n_core_failed_checks = 0`.

## Tiêu chí FAIL cần hiểu đúng
Nếu notebook này báo `FAIL`, điều đó **không nhất thiết** có nghĩa model upstream sai.  
Nó thường có nghĩa:
- thiếu artifact đầu ra,
- summary JSON chưa khớp,
- đường dẫn lưu file upstream chưa đúng,
- hoặc payload inference của một nhánh chưa sẵn sàng.

## Đầu ra cuối của file 08
Notebook này sẽ lưu các file audit/snapshot riêng cho integration như:
- `integration_file_registry.csv`
- `integration_dependency_contract.csv`
- `integration_object_audit.csv`
- `integration_schema_audit.csv`
- `integration_ui_payload_preview.json`
- `integration_ui_payload_index.csv`
- `integration_check_summary.csv`
- `integration_check_summary.json`

> Mục tiêu của notebook này là làm **cầu nối cuối từ artifact notebook sang Streamlit**, bám đúng hướng kiến trúc trong báo cáo và rubric: raw data → processed data → trained artifacts → web UI. 

## Cell 1: Import thư viện cần thiết

**Mục tiêu**
- Import đầy đủ thư viện để:
  - đọc JSON / CSV / Parquet / pickle / joblib,
  - audit object,
  - chuẩn hóa dữ liệu,
  - hiển thị bảng kiểm tra trong notebook.

**Vì sao cell này quan trọng**
- Notebook 08 là notebook integration, nên cần hỗ trợ **nhiều loại artifact khác nhau**:
  - bảng dữ liệu processed,
  - model sklearn,
  - bundle pickle của Surprise,
  - file summary JSON,
  - payload preview cho UI.

**Đầu ra mong đợi**
- môi trường import ổn định,
- không lỗi thiếu thư viện nền,
- có cờ `SURPRISE_AVAILABLE` để kiểm tra môi trường recommendation trước khi load pickle Surprise.


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import pickle
import math
import ast

import numpy as np
import pandas as pd
import joblib

from IPython.display import display, Markdown

try:
    from surprise import AlgoBase  # chỉ để đảm bảo môi trường có scikit-surprise khi load pickle
    SURPRISE_AVAILABLE = True
except Exception:
    SURPRISE_AVAILABLE = False

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

## Cell 2: Khai báo cấu hình thư mục và định vị project

**Mục tiêu**
- Xác định đúng thư mục gốc của project để notebook 08 có thể chạy trong nhiều vị trí khác nhau.
- Tách rõ các thư mục:
  - `data/processed`
  - `artifacts/models`
  - `artifacts/metrics`
  - `artifacts/plots`
  - `artifacts/predictions`

**Vì sao cell này quan trọng**
- File 08 đọc output từ nhiều notebook khác nhau.
- Nếu resolve path không ổn định, integration check sẽ fail dù artifact upstream thực ra đã tồn tại.

**Tiêu chí tốt của cell này**
- không hard-code tuyệt đối vào một máy duy nhất,
- ưu tiên cấu trúc project thực tế,
- vẫn fallback mềm để notebook không vỡ ngay từ đầu.

**Đầu ra mong đợi**
- `BASE_DIR`
- các thư mục processed / artifacts
- bảo đảm thư mục `METRIC_DIR` tồn tại để lưu audit và summary integration.


In [2]:
# =========================
# CONFIG
# =========================
def locate_project_base():
    candidates = [
        Path("."),
        Path(".."),
        Path("../.."),
    ]
    for base in candidates:
        if (base / "data" / "processed").exists():
            return base.resolve()
    # fallback mềm: vẫn trả về current dir để notebook không crash ngay từ đầu
    return Path(".").resolve()

BASE_DIR = locate_project_base()
NOTEBOOK_DIR_CANDIDATES = [
    (BASE_DIR / "notebooks").resolve(),
    BASE_DIR.resolve(),
    Path(".").resolve(),
]

PROCESSED_DIR = BASE_DIR / "data" / "processed"
ARTIFACT_DIR = BASE_DIR / "artifacts"

MODEL_DIR = ARTIFACT_DIR / "models"
METRIC_DIR = ARTIFACT_DIR / "metrics"
PLOT_DIR = ARTIFACT_DIR / "plots"
PRED_DIR = ARTIFACT_DIR / "predictions"
DATA_ARTIFACT_DIR = ARTIFACT_DIR / "data"

for folder in [PROCESSED_DIR, MODEL_DIR, METRIC_DIR, PLOT_DIR, PRED_DIR, DATA_ARTIFACT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("BASE_DIR       :", BASE_DIR)
print("PROCESSED_DIR  :", PROCESSED_DIR)
print("ARTIFACT_DIR   :", ARTIFACT_DIR)
print("NOTEBOOK_DIRS  :", NOTEBOOK_DIR_CANDIDATES)

INTEGRATION_NOTEBOOK_NAME = "08_integration_check.ipynb"
UPSTREAM_NOTEBOOKS = [
    "02_data_preprocessing_master_dataset.ipynb",
    "03_classification_pipeline.ipynb",
    "04_regression_pipeline.ipynb",
    "05_clustering_rfm.ipynb",
    "06_surprise_recommendation.ipynb",
    "07_fpgrowth_rules.ipynb",
]
OPTIONAL_NON_DEPENDENCY_NOTEBOOKS = [
    "01_eda_data_structure_check.ipynb",
    "09_chi_square_quick_check.ipynb",
]


BASE_DIR       : C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project
PROCESSED_DIR  : C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project\data\processed
ARTIFACT_DIR   : C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project\artifacts
NOTEBOOK_DIRS  : [WindowsPath('C:/Users/trant/OneDrive/Tài liệu/Documents/UTE/BIGDATA/olist_ml_project/notebooks'), WindowsPath('C:/Users/trant/OneDrive/Tài liệu/Documents/UTE/BIGDATA/olist_ml_project'), WindowsPath('C:/Users/trant/OneDrive/Tài liệu/Documents/UTE/BIGDATA/olist_ml_project/notebooks')]


## Cell 3: Định nghĩa các hàm helper dùng cho integration check

**Mục tiêu**
- Gom tất cả helper dùng chung vào một cell duy nhất để:
  - audit file,
  - audit object,
  - audit schema,
  - chuẩn hóa path,
  - load artifact an toàn,
  - chuyển object sang JSON-serializable.

**Nhóm helper chính**
1. **Audit helpers**
   - `add_check`
   - `add_object_audit`
   - `add_schema_audit`

2. **Path / IO helpers**
   - `resolve_saved_path`
   - `safe_read_json`
   - `safe_joblib_load`
   - `safe_pickle_load`
   - `safe_read_table`

3. **Serialization / utility helpers**
   - `unique_preserve_order`
   - `normalize_text`
   - `make_json_serializable`
   - `first_non_null_value`

**Vì sao cell này quan trọng**
- Nếu không có lớp helper này, phần code của notebook 08 sẽ dài, lặp và khó bảo trì.
- Đây là cell bảo đảm notebook integration **fail đúng chỗ** thay vì fail mơ hồ.

**Đầu ra mong đợi**
- Có đủ helper để các cell sau chỉ tập trung vào logic integration.


In [3]:
# =========================
# HELPERS
# =========================
file_registry_rows = []
object_audit_rows = []
schema_audit_rows = []
check_rows = []
ui_payload_preview = {}

def add_check(module, check_name, passed, notes=None, severity="core"):
    check_rows.append({
        "module": module,
        "check_name": check_name,
        "passed": bool(passed),
        "severity": severity,
        "notes": "" if notes is None else str(notes)
    })

def add_object_audit(module, object_name, loaded, object_type=None, notes=None):
    object_audit_rows.append({
        "module": module,
        "object_name": object_name,
        "loaded": bool(loaded),
        "object_type": "" if object_type is None else str(object_type),
        "notes": "" if notes is None else str(notes)
    })

def add_schema_audit(module, schema_name, expected_columns, provided_columns):
    expected_columns = list(expected_columns) if expected_columns is not None else []
    provided_columns = list(provided_columns) if provided_columns is not None else []
    missing_cols = [c for c in expected_columns if c not in provided_columns]
    extra_cols = [c for c in provided_columns if c not in expected_columns]

    schema_audit_rows.append({
        "module": module,
        "schema_name": schema_name,
        "n_expected": len(expected_columns),
        "n_provided": len(provided_columns),
        "n_missing": len(missing_cols),
        "n_extra": len(extra_cols),
        "missing_columns": ", ".join(map(str, missing_cols[:50])),
        "extra_columns": ", ".join(map(str, extra_cols[:50]))
    })

    return missing_cols, extra_cols

def unique_preserve_order(items):
    seen = set()
    out = []
    for item in items:
        if item not in seen:
            out.append(item)
            seen.add(item)
    return out

def resolve_saved_path(path_like):
    """
    Xử lý path lưu trong summary JSON / bundle:
    - hỗ trợ Windows backslash
    - hỗ trợ path tương đối kiểu '..\\artifacts\\...'
    - thử resolve theo nhiều gốc khác nhau
    """
    if path_like is None:
        return None

    if isinstance(path_like, float) and pd.isna(path_like):
        return None

    raw = str(path_like).strip()
    if raw == "" or raw.lower() == "none":
        return None

    raw = raw.replace("\\", "/")
    p = Path(raw)

    if p.is_absolute():
        return p.resolve()

    candidate_refs = []
    candidate_refs.extend(NOTEBOOK_DIR_CANDIDATES)

    # thêm 1 số gốc thường gặp
    candidate_refs.extend([
        BASE_DIR.parent.resolve(),
        (BASE_DIR / "app").resolve(),
    ])

    candidate_refs = unique_preserve_order(candidate_refs)

    for ref in candidate_refs:
        try:
            cand = (ref / p).resolve()
            if cand.exists():
                return cand
        except Exception:
            pass

    # fallback cuối cùng
    return (BASE_DIR / p).resolve()

def safe_read_json(path_like):
    path_obj = resolve_saved_path(path_like)
    if path_obj is None or not path_obj.exists():
        return None, path_obj
    try:
        with open(path_obj, "r", encoding="utf-8") as f:
            return json.load(f), path_obj
    except Exception as e:
        return {"_error": str(e)}, path_obj

def safe_joblib_load(path_like):
    path_obj = resolve_saved_path(path_like)
    if path_obj is None or not path_obj.exists():
        return None, path_obj, "file_not_found"
    try:
        obj = joblib.load(path_obj)
        return obj, path_obj, None
    except Exception as e:
        return None, path_obj, str(e)

def safe_pickle_load(path_like):
    path_obj = resolve_saved_path(path_like)
    if path_obj is None or not path_obj.exists():
        return None, path_obj, "file_not_found"
    try:
        with open(path_obj, "rb") as f:
            obj = pickle.load(f)
        return obj, path_obj, None
    except Exception as e:
        return None, path_obj, str(e)

def safe_read_table(parquet_path=None, csv_path=None, preferred_label=None):
    parquet_obj = resolve_saved_path(parquet_path) if parquet_path is not None else None
    csv_obj = resolve_saved_path(csv_path) if csv_path is not None else None

    if parquet_obj is not None and parquet_obj.exists():
        try:
            return pd.read_parquet(parquet_obj), parquet_obj, None
        except Exception as e:
            return None, parquet_obj, f"read_parquet_failed: {e}"

    if csv_obj is not None and csv_obj.exists():
        try:
            return pd.read_csv(csv_obj), csv_obj, None
        except Exception as e:
            return None, csv_obj, f"read_csv_failed: {e}"

    label = preferred_label if preferred_label is not None else "table"
    return None, parquet_obj or csv_obj, f"{label}_not_found"

def normalize_text(value, default="unknown"):
    if pd.isna(value):
        return default
    s = str(value).strip()
    return default if s == "" else s

def make_json_serializable(obj):
    if isinstance(obj, dict):
        return {str(k): make_json_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_serializable(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, Path):
        return str(obj)
    return obj

def first_non_null_value(series, default=None):
    s = series.dropna()
    if len(s) == 0:
        return default
    return s.iloc[0]

def exists_in_registry(file_key, registry_df):
    if registry_df is None or len(registry_df) == 0:
        return False
    subset = registry_df.loc[registry_df["file_key"].eq(file_key), "exists"]
    return bool(subset.any())

def note_missing_vs_empty(df_obj, read_error, not_found_token):
    if df_obj is None:
        return not_found_token if read_error is None else str(read_error)
    if hasattr(df_obj, "__len__") and len(df_obj) == 0:
        return "loaded_but_empty"
    return "loaded"

def build_dependency_contract_rows(registry_dict, registry_df):
    dependency_specs = [
        # notebook 02
        ("02_data_preprocessing_master_dataset.ipynb", "orders_base_final_parquet", "processed_data", "classification/regression input table", "core"),
        ("02_data_preprocessing_master_dataset.ipynb", "rfm_df_parquet", "processed_data", "clustering input table", "core"),
        ("02_data_preprocessing_master_dataset.ipynb", "ratings_df_parquet", "processed_data", "recommendation input table", "support"),
        ("02_data_preprocessing_master_dataset.ipynb", "transactions_df_parquet", "processed_data", "FP-Growth input basket table", "support"),
        ("02_data_preprocessing_master_dataset.ipynb", "transactions_order_category_df_parquet", "processed_data", "transaction-category diagnostic table", "support"),
        ("02_data_preprocessing_master_dataset.ipynb", "product_lookup_parquet", "processed_data", "product metadata lookup", "support"),
        # notebook 03
        ("03_classification_pipeline.ipynb", "classification_summary_json", "classification", "summary contract for classifier artifacts", "support"),
        ("03_classification_pipeline.ipynb", "best_classifier_model", "classification", "tabular classifier deployment model", "core"),
        ("03_classification_pipeline.ipynb", "best_text_classifier_model", "classification", "text classifier deployment model", "support"),
        # notebook 04
        ("04_regression_pipeline.ipynb", "regression_summary_json", "regression", "summary contract for regressor artifact", "support"),
        ("04_regression_pipeline.ipynb", "best_regressor_model", "regression", "regressor deployment model", "core"),
        # notebook 05
        ("05_clustering_rfm.ipynb", "clustering_summary_json", "clustering", "summary contract for clustering artifacts", "support"),
        ("05_clustering_rfm.ipynb", "kmeans_model", "clustering", "KMeans inference model", "support"),
        ("05_clustering_rfm.ipynb", "gmm_model", "clustering", "GMM inference model", "support"),
        ("05_clustering_rfm.ipynb", "rfm_scaler_model", "clustering", "shared scaler for clustering inference", "core"),
        ("05_clustering_rfm.ipynb", "rfm_pca_model", "clustering", "PCA projection helper for clustering", "support"),
        ("05_clustering_rfm.ipynb", "kmeans_profile_csv", "clustering", "cluster profile table for segment mapping", "support"),
        ("05_clustering_rfm.ipynb", "gmm_profile_csv", "clustering", "cluster profile table for segment mapping", "support"),
        # notebook 06
        ("06_surprise_recommendation.ipynb", "surprise_summary_json", "recommendation", "summary contract for recommendation artifacts", "support"),
        ("06_surprise_recommendation.ipynb", "best_surprise_model", "recommendation", "primary Surprise model", "core"),
        ("06_surprise_recommendation.ipynb", "seen_items_map", "recommendation", "seen-item memory for filtering recommendations", "core"),
        ("06_surprise_recommendation.ipynb", "surprise_deployment_bundle", "recommendation", "bundle for recommendation defaults", "core"),
        ("06_surprise_recommendation.ipynb", "item_knn_neighbors_model", "recommendation", "optional item-neighbor model", "support"),
        ("06_surprise_recommendation.ipynb", "rec_product_lookup_csv", "recommendation", "metadata lookup for recommendation payload", "support"),
        ("06_surprise_recommendation.ipynb", "known_users_csv", "recommendation", "known user list for smoke test", "support"),
        ("06_surprise_recommendation.ipynb", "known_items_csv", "recommendation", "known item list for smoke test", "support"),
        ("06_surprise_recommendation.ipynb", "candidate_items_csv", "recommendation", "candidate item universe", "support"),
        ("06_surprise_recommendation.ipynb", "popular_products_fallback_csv", "recommendation", "cold-start fallback table", "support"),
        # notebook 07
        ("07_fpgrowth_rules.ipynb", "fpgrowth_summary_json", "fpgrowth", "summary contract for FP-Growth outputs", "support"),
        ("07_fpgrowth_rules.ipynb", "frequent_itemsets_csv", "fpgrowth", "main frequent itemsets table", "core"),
        ("07_fpgrowth_rules.ipynb", "association_rules_csv", "fpgrowth", "main association rules table for UI", "core"),
        ("07_fpgrowth_rules.ipynb", "fpgrowth_threshold_search_csv", "fpgrowth", "threshold search diagnostics", "support"),
        ("07_fpgrowth_rules.ipynb", "top_rules_preview_csv", "fpgrowth", "preview rules for UI/demo fallback", "support"),
        ("07_fpgrowth_rules.ipynb", "top_rules_report_csv", "fpgrowth", "report-ready rules table", "support"),
        ("07_fpgrowth_rules.ipynb", "top_itemsets_preview_csv", "fpgrowth", "preview itemsets fallback", "support"),
        ("07_fpgrowth_rules.ipynb", "transaction_item_frequency_csv", "fpgrowth", "item frequency diagnostics", "support"),
    ]

    rows = []
    exists_map = {}
    if registry_df is not None and len(registry_df) > 0:
        exists_map = dict(zip(registry_df["file_key"], registry_df["exists"]))

    for upstream_notebook, registry_key, module_name, purpose, required_level in dependency_specs:
        raw_path = registry_dict.get(registry_key)
        path_obj = resolve_saved_path(raw_path)
        rows.append({
            "upstream_notebook": upstream_notebook,
            "module": module_name,
            "registry_key": registry_key,
            "purpose": purpose,
            "required_level": required_level,
            "path": "" if path_obj is None else str(path_obj),
            "exists": bool(exists_map.get(registry_key, False)),
        })
    return rows


## Cell 4: Khai báo file registry mặc định và đọc summary JSON

**Mục tiêu**
- Tạo một **registry mặc định** cho toàn bộ file mà notebook 08 cần đọc.
- Đọc toàn bộ summary JSON từ các notebook upstream để:
  - biết artifact nào được tạo thật,
  - biết notebook upstream đang lưu file ở đâu,
  - cho phép notebook 08 override lại đường dẫn nếu cần.

**Phân nhóm registry**
- **Processed tables**: đầu ra từ notebook 02
- **Summary JSON**: đầu ra tổng hợp từ notebook 03–07
- **Core models / bundles**: model chính dùng cho inference
- **Recommendation helper files**: known users/items/candidates/fallback
- **Clustering profiles**: profile cluster để map cluster → segment
- **FP-Growth outputs**: itemsets, rules, threshold search, preview files

**Đầu ra mong đợi**
- `registry` mặc định đủ mạnh để chạy cả khi chưa override
- `classification_summary`, `regression_summary`, `clustering_summary`, `surprise_summary`, `fpgrowth_summary`
- bảng trạng thái load của summary JSON

**Lưu ý**
- Summary JSON không thay thế artifact chính; nó chỉ giúp notebook 08 hiểu đúng contract của upstream notebook.


In [4]:
# =========================
# DEFAULT FILE REGISTRY
# =========================
registry = {
    # processed tables
    "orders_base_final_parquet": PROCESSED_DIR / "orders_base_final.parquet",
    "orders_base_final_csv": PROCESSED_DIR / "orders_base_final.csv",
    "rfm_df_parquet": PROCESSED_DIR / "rfm_df.parquet",
    "rfm_df_csv": PROCESSED_DIR / "rfm_df.csv",
    "ratings_df_parquet": PROCESSED_DIR / "ratings_df.parquet",
    "ratings_df_csv": PROCESSED_DIR / "ratings_df.csv",
    "transactions_df_parquet": PROCESSED_DIR / "transactions_df.parquet",
    "transactions_df_csv": PROCESSED_DIR / "transactions_df.csv",
    "transactions_order_category_df_parquet": PROCESSED_DIR / "transactions_order_category_df.parquet",
    "transactions_order_category_df_csv": PROCESSED_DIR / "transactions_order_category_df.csv",
    "product_lookup_parquet": PROCESSED_DIR / "product_lookup.parquet",
    "product_lookup_csv": PROCESSED_DIR / "product_lookup.csv",

    # summary files
    "classification_summary_json": METRIC_DIR / "classification_final_summary.json",
    "regression_summary_json": METRIC_DIR / "regression_final_summary.json",
    "clustering_summary_json": METRIC_DIR / "clustering_final_summary.json",
    "surprise_summary_json": METRIC_DIR / "surprise_final_summary.json",
    "fpgrowth_summary_json": METRIC_DIR / "fpgrowth_final_summary.json",

    # core models
    "best_classifier_model": MODEL_DIR / "best_classifier_baseline.joblib",
    "best_text_classifier_model": MODEL_DIR / "best_classifier_text_tfidf.joblib",
    "best_regressor_model": MODEL_DIR / "best_regressor_baseline.joblib",
    "kmeans_model": MODEL_DIR / "kmeans_model.joblib",
    "gmm_model": MODEL_DIR / "gmm_model.joblib",
    "rfm_scaler_model": MODEL_DIR / "rfm_standard_scaler.joblib",
    "rfm_pca_model": MODEL_DIR / "rfm_pca_2d.joblib",
    "best_surprise_model": MODEL_DIR / "best_surprise_model.pkl",
    "seen_items_map": MODEL_DIR / "seen_items_map.pkl",
    "surprise_deployment_bundle": MODEL_DIR / "surprise_deployment_bundle.pkl",
    "item_knn_neighbors_model": MODEL_DIR / "item_knn_neighbors_model.pkl",

    # recommendation data artifacts
    "rec_product_lookup_csv": DATA_ARTIFACT_DIR / "product_lookup.csv",
    "known_users_csv": DATA_ARTIFACT_DIR / "known_users.csv",
    "known_items_csv": DATA_ARTIFACT_DIR / "known_items.csv",
    "candidate_items_csv": DATA_ARTIFACT_DIR / "candidate_items.csv",
    "popular_products_fallback_csv": PRED_DIR / "popular_products_fallback.csv",

    # clustering profiles
    "kmeans_profile_csv": METRIC_DIR / "kmeans_cluster_profile.csv",
    "gmm_profile_csv": METRIC_DIR / "gmm_cluster_profile.csv",

    # FP-Growth outputs
    "frequent_itemsets_csv": METRIC_DIR / "frequent_itemsets.csv",
    "association_rules_csv": METRIC_DIR / "association_rules.csv",
    "fpgrowth_threshold_search_csv": METRIC_DIR / "fpgrowth_threshold_search.csv",
    "top_rules_preview_csv": PRED_DIR / "top_association_rules_preview.csv",
    "top_rules_report_csv": METRIC_DIR / "top_association_rules_report.csv",
    "top_itemsets_preview_csv": PRED_DIR / "top_frequent_itemsets_preview.csv",
    "transaction_item_frequency_csv": METRIC_DIR / "transaction_item_frequency.csv",
}

# -------------------------
# READ ALL SUMMARY JSONs
# -------------------------
classification_summary, classification_summary_path = safe_read_json(registry["classification_summary_json"])
regression_summary, regression_summary_path = safe_read_json(registry["regression_summary_json"])
clustering_summary, clustering_summary_path = safe_read_json(registry["clustering_summary_json"])
surprise_summary, surprise_summary_path = safe_read_json(registry["surprise_summary_json"])
fpgrowth_summary, fpgrowth_summary_path = safe_read_json(registry["fpgrowth_summary_json"])

summary_map = {
    "classification_summary": classification_summary,
    "regression_summary": regression_summary,
    "clustering_summary": clustering_summary,
    "surprise_summary": surprise_summary,
    "fpgrowth_summary": fpgrowth_summary,
}

display(Markdown("### Trạng thái summary JSON"))
display(pd.DataFrame([
    {"summary_name": k, "loaded": isinstance(v, dict) and "_error" not in v and v is not None}
    for k, v in summary_map.items()
]))

### Trạng thái summary JSON

,summary_name,loaded
0,classification_summary,True
1,regression_summary,True
2,clustering_summary,True
3,surprise_summary,True
4,fpgrowth_summary,True


## Cell 5: Cập nhật registry từ summary và kiểm tra file artifact cốt lõi

**Mục tiêu**
- Nếu summary JSON upstream có ghi đường dẫn artifact thực tế, notebook 08 sẽ dùng các đường dẫn đó làm **source of truth**.
- Sau khi cập nhật registry, notebook sẽ:
  - audit tồn tại của từng file,
  - tạo bảng file registry,
  - xác định **core files** có đủ chưa.

**Vì sao cell này quan trọng**
- Trong dự án thật, artifact có thể được lưu từ path tương đối khác nhau.
- Nếu chỉ bám registry mặc định cứng, notebook 08 dễ fail giả.

**Đầu ra mong đợi**
- `file_registry_df`
- `integration_file_registry.csv`
- `integration_dependency_contract.csv`
- check `core_required_files_exist`

**Nguyên tắc chấm PASS**
- Notebook 08 chỉ nên PASS khi các artifact lõi thật sự tồn tại.
- Không được “ép PASS giả” bằng cách bỏ qua artifact cốt lõi bị thiếu.


In [5]:
# =========================
# OVERRIDE REGISTRY FROM SUMMARIES / BUNDLE-LIKE METADATA
# =========================
if isinstance(classification_summary, dict) and "_error" not in classification_summary:
    if classification_summary.get("best_baseline_model_file"):
        registry["best_classifier_model"] = resolve_saved_path(classification_summary["best_baseline_model_file"])
    if classification_summary.get("best_text_model_file"):
        registry["best_text_classifier_model"] = resolve_saved_path(classification_summary["best_text_model_file"])

if isinstance(regression_summary, dict) and "_error" not in regression_summary:
    if regression_summary.get("best_model_file"):
        registry["best_regressor_model"] = resolve_saved_path(regression_summary["best_model_file"])

if isinstance(clustering_summary, dict) and "_error" not in clustering_summary:
    saved_files = clustering_summary.get("saved_files", {})
    if isinstance(saved_files, dict):
        key_map = {
            "kmeans_model": "kmeans_model",
            "gmm_model": "gmm_model",
            "scaler": "rfm_scaler_model",
            "pca_2d": "rfm_pca_model",
            "kmeans_profile": "kmeans_profile_csv",
            "gmm_profile": "gmm_profile_csv",
        }
        for old_key, registry_key in key_map.items():
            if saved_files.get(old_key):
                registry[registry_key] = resolve_saved_path(saved_files[old_key])

if isinstance(surprise_summary, dict) and "_error" not in surprise_summary:
    direct_key_map = {
        "best_surprise_model": "best_surprise_model",
        "surprise_deployment_bundle": "surprise_deployment_bundle",
        "known_users_csv": "known_users_csv",
        "known_items_csv": "known_items_csv",
        "candidate_items_csv": "candidate_items_csv",
        "product_lookup_csv": "rec_product_lookup_csv",
    }
    for summary_key, registry_key in direct_key_map.items():
        if surprise_summary.get(summary_key):
            registry[registry_key] = resolve_saved_path(surprise_summary[summary_key])

    integration_files = surprise_summary.get("files_expected_by_integration", {})
    if isinstance(integration_files, dict):
        integration_key_map = {
            "best_surprise_model": "best_surprise_model",
            "surprise_deployment_bundle": "surprise_deployment_bundle",
            "known_users_csv": "known_users_csv",
            "known_items_csv": "known_items_csv",
            "candidate_items_csv": "candidate_items_csv",
            "product_lookup_csv": "rec_product_lookup_csv",
        }
        for summary_key, registry_key in integration_key_map.items():
            if integration_files.get(summary_key):
                registry[registry_key] = resolve_saved_path(integration_files[summary_key])

if isinstance(fpgrowth_summary, dict) and "_error" not in fpgrowth_summary:
    saved_files = fpgrowth_summary.get("saved_files", {})
    if isinstance(saved_files, dict):
        key_map = {
            "frequent_itemsets": "frequent_itemsets_csv",
            "association_rules": "association_rules_csv",
            "threshold_search": "fpgrowth_threshold_search_csv",
            "top_rules_preview": "top_rules_preview_csv",
            "top_rules_report": "top_rules_report_csv",
            "top_itemsets_preview": "top_itemsets_preview_csv",
            "item_frequency": "transaction_item_frequency_csv",
        }
        for old_key, registry_key in key_map.items():
            if saved_files.get(old_key):
                registry[registry_key] = resolve_saved_path(saved_files[old_key])

# -------------------------
# FILE REGISTRY AUDIT
# -------------------------
file_registry_rows = []
for file_key, file_path in registry.items():
    path_obj = resolve_saved_path(file_path)
    file_registry_rows.append({
        "file_key": file_key,
        "path": str(path_obj) if path_obj is not None else "",
        "exists": bool(path_obj is not None and path_obj.exists())
    })

file_registry_df = pd.DataFrame(file_registry_rows).sort_values(["exists", "file_key"], ascending=[False, True]).reset_index(drop=True)

display(Markdown("### File registry"))
display(file_registry_df)

core_required = [
    "orders_base_final_parquet",
    "rfm_df_parquet",
    "best_classifier_model",
    "best_regressor_model",
    "rfm_scaler_model",
    "best_surprise_model",
    "seen_items_map",
    "surprise_deployment_bundle",
    "association_rules_csv",
    "frequent_itemsets_csv",
]

missing_core = [k for k in core_required if not exists_in_registry(k, file_registry_df)]
add_check(
    module="registry",
    check_name="core_required_files_exist",
    passed=(len(missing_core) == 0),
    notes="; ".join(missing_core) if missing_core else "All core files found"
)

# -------------------------
# DEPENDENCY CONTRACT SNAPSHOT
# -------------------------
dependency_contract_df = pd.DataFrame(
    build_dependency_contract_rows(registry, file_registry_df)
).sort_values(["upstream_notebook", "module", "required_level", "registry_key"]).reset_index(drop=True)

display(Markdown("### Dependency contract"))
display(dependency_contract_df)

file_registry_output_path = METRIC_DIR / "integration_file_registry.csv"
dependency_contract_output_path = METRIC_DIR / "integration_dependency_contract.csv"

file_registry_df.to_csv(file_registry_output_path, index=False)
dependency_contract_df.to_csv(dependency_contract_output_path, index=False)

print("Đã lưu:", file_registry_output_path)
print("Đã lưu:", dependency_contract_output_path)


### File registry

,file_key,path,exists
0,association_rules_csv,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
1,best_classifier_model,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
2,best_regressor_model,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
3,best_surprise_model,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
4,best_text_classifier_model,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
5,candidate_items_csv,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
6,classification_summary_json,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
7,clustering_summary_json,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
8,fpgrowth_summary_json,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
9,fpgrowth_threshold_search_csv,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True


### Dependency contract

,upstream_notebook,module,registry_key,purpose,required_level,path,exists
0,02_data_preprocessing_master_dataset.ipynb,processed_data,orders_base_final_parquet,classification/regression input table,core,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
1,02_data_preprocessing_master_dataset.ipynb,processed_data,rfm_df_parquet,clustering input table,core,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
2,02_data_preprocessing_master_dataset.ipynb,processed_data,product_lookup_parquet,product metadata lookup,support,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
3,02_data_preprocessing_master_dataset.ipynb,processed_data,ratings_df_parquet,recommendation input table,support,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
4,02_data_preprocessing_master_dataset.ipynb,processed_data,transactions_df_parquet,FP-Growth input basket table,support,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
5,02_data_preprocessing_master_dataset.ipynb,processed_data,transactions_order_category_df_parquet,transaction-category diagnostic table,support,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
6,03_classification_pipeline.ipynb,classification,best_classifier_model,tabular classifier deployment model,core,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
7,03_classification_pipeline.ipynb,classification,best_text_classifier_model,text classifier deployment model,support,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
8,03_classification_pipeline.ipynb,classification,classification_summary_json,summary contract for classifier artifacts,support,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True
9,04_regression_pipeline.ipynb,regression,best_regressor_model,regressor deployment model,core,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,True


Đã lưu: C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project\artifacts\metrics\integration_file_registry.csv
Đã lưu: C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project\artifacts\metrics\integration_dependency_contract.csv


## Cell 6: Load các bảng dữ liệu processed

**Mục tiêu**
- Đọc các bảng đầu vào đã được chuẩn hóa ở notebook 02 để dùng cho:
  - schema audit,
  - sample inference,
  - payload preview.

**Các bảng được đọc**
- `orders_base_final`
- `rfm_df`
- `ratings_df`
- `transactions_df`
- `transactions_order_category_df`
- `product_lookup_processed`

**Vai trò của từng bảng**
- `orders_base_final`: nguồn inference cho classification / regression
- `rfm_df`: nguồn inference cho clustering
- `ratings_df`: bằng chứng dữ liệu recommendation đã sẵn
- `transactions_df`: bằng chứng dữ liệu basket mining đã sẵn
- `product_lookup_processed`: metadata sản phẩm hỗ trợ recommendation UI

**Đầu ra mong đợi**
- bảng summary load thành công/thất bại cho processed tables
- check tải dữ liệu tối thiểu cho integration


In [6]:
# =========================
# LOAD PROCESSED TABLES
# =========================
orders_base_final, orders_base_final_path, err_orders = safe_read_table(
    registry["orders_base_final_parquet"], registry["orders_base_final_csv"], preferred_label="orders_base_final"
)
rfm_df, rfm_df_path, err_rfm = safe_read_table(
    registry["rfm_df_parquet"], registry["rfm_df_csv"], preferred_label="rfm_df"
)
ratings_df, ratings_df_path, err_ratings = safe_read_table(
    registry["ratings_df_parquet"], registry["ratings_df_csv"], preferred_label="ratings_df"
)
transactions_df, transactions_df_path, err_tx = safe_read_table(
    registry["transactions_df_parquet"], registry["transactions_df_csv"], preferred_label="transactions_df"
)
transactions_order_category_df, transactions_order_category_df_path, err_tx_order = safe_read_table(
    registry["transactions_order_category_df_parquet"], registry["transactions_order_category_df_csv"], preferred_label="transactions_order_category_df"
)
processed_product_lookup_df, processed_product_lookup_path, err_lookup = safe_read_table(
    registry["product_lookup_parquet"], registry["product_lookup_csv"], preferred_label="product_lookup"
)

processed_tables_summary = pd.DataFrame([
    {
        "table_name": "orders_base_final",
        "loaded": orders_base_final is not None,
        "shape": None if orders_base_final is None else orders_base_final.shape,
        "source": "" if orders_base_final_path is None else str(orders_base_final_path),
        "error": "" if err_orders is None else err_orders
    },
    {
        "table_name": "rfm_df",
        "loaded": rfm_df is not None,
        "shape": None if rfm_df is None else rfm_df.shape,
        "source": "" if rfm_df_path is None else str(rfm_df_path),
        "error": "" if err_rfm is None else err_rfm
    },
    {
        "table_name": "ratings_df",
        "loaded": ratings_df is not None,
        "shape": None if ratings_df is None else ratings_df.shape,
        "source": "" if ratings_df_path is None else str(ratings_df_path),
        "error": "" if err_ratings is None else err_ratings
    },
    {
        "table_name": "transactions_df",
        "loaded": transactions_df is not None,
        "shape": None if transactions_df is None else transactions_df.shape,
        "source": "" if transactions_df_path is None else str(transactions_df_path),
        "error": "" if err_tx is None else err_tx
    },
    {
        "table_name": "transactions_order_category_df",
        "loaded": transactions_order_category_df is not None,
        "shape": None if transactions_order_category_df is None else transactions_order_category_df.shape,
        "source": "" if transactions_order_category_df_path is None else str(transactions_order_category_df_path),
        "error": "" if err_tx_order is None else err_tx_order
    },
    {
        "table_name": "product_lookup_processed",
        "loaded": processed_product_lookup_df is not None,
        "shape": None if processed_product_lookup_df is None else processed_product_lookup_df.shape,
        "source": "" if processed_product_lookup_path is None else str(processed_product_lookup_path),
        "error": "" if err_lookup is None else err_lookup
    },
])

display(Markdown("### Processed tables summary"))
display(processed_tables_summary)

for row in processed_tables_summary.itertuples(index=False):
    add_check(
        module="processed_data",
        check_name=f"{row.table_name}_loaded",
        passed=bool(row.loaded),
        notes=row.error if row.error else row.source,
        severity="core" if row.table_name in {"orders_base_final", "rfm_df"} else "support"
    )

### Processed tables summary

,table_name,loaded,shape,source,error
0,orders_base_final,True,"(99441, 52)",C:\Users\trant\OneDrive\Tài liệu\Documents\U...,
1,rfm_df,True,"(93358, 4)",C:\Users\trant\OneDrive\Tài liệu\Documents\U...,
2,ratings_df,True,"(101198, 4)",C:\Users\trant\OneDrive\Tài liệu\Documents\U...,
3,transactions_df,True,"(96478, 2)",C:\Users\trant\OneDrive\Tài liệu\Documents\U...,
4,transactions_order_category_df,True,"(96478, 6)",C:\Users\trant\OneDrive\Tài liệu\Documents\U...,
5,product_lookup_processed,True,"(32789, 6)",C:\Users\trant\OneDrive\Tài liệu\Documents\U...,


## Cell 7: Load model, bundle và artifact từ các module trước

**Mục tiêu**
- Load toàn bộ artifact inference quan trọng từ các notebook 03–07.

**Nhóm artifact chính**
1. **Classification**
   - model tabular
   - model text

2. **Regression**
   - model deployment

3. **Clustering**
   - `kmeans_model`
   - `gmm_model`
   - `rfm_scaler`
   - `rfm_pca`

4. **Recommendation**
   - `best_surprise_model`
   - `seen_items_map`
   - `surprise_deployment_bundle`
   - `item_knn_neighbors_model`
   - helper tables cho known users/items/candidates/fallback

5. **FP-Growth**
   - `frequent_itemsets.csv`
   - `association_rules.csv`
   - `fpgrowth_threshold_search.csv`
   - preview/report files nếu có

**Vì sao cell này quan trọng**
- Đây là cell xác nhận dự án có thật sự bước được từ “train xong” sang “load để suy luận” hay không.
- Nếu model không load lại được, UI không thể hoạt động ổn định.

**Đầu ra mong đợi**
- `integration_object_audit.csv`
- bảng object audit với trạng thái `loaded`, `object_type`, `notes`
- check riêng cho từng artifact quan trọng


In [7]:
# =========================
# LOAD MODEL / BUNDLE ARTIFACTS
# =========================
classifier_model, classifier_model_path, classifier_err = safe_joblib_load(registry["best_classifier_model"])
text_classifier_model, text_classifier_model_path, text_classifier_err = safe_joblib_load(registry["best_text_classifier_model"])
regressor_model, regressor_model_path, regressor_err = safe_joblib_load(registry["best_regressor_model"])

kmeans_model, kmeans_model_path, kmeans_err = safe_joblib_load(registry["kmeans_model"])
gmm_model, gmm_model_path, gmm_err = safe_joblib_load(registry["gmm_model"])
rfm_scaler, rfm_scaler_path, rfm_scaler_err = safe_joblib_load(registry["rfm_scaler_model"])
rfm_pca, rfm_pca_path, rfm_pca_err = safe_joblib_load(registry["rfm_pca_model"])

surprise_model, surprise_model_path, surprise_err = safe_pickle_load(registry["best_surprise_model"])
seen_items_map, seen_items_map_path, seen_items_map_err = safe_pickle_load(registry["seen_items_map"])
surprise_bundle, surprise_bundle_path, surprise_bundle_err = safe_pickle_load(registry["surprise_deployment_bundle"])
item_knn_model, item_knn_model_path, item_knn_err = safe_pickle_load(registry["item_knn_neighbors_model"])

rec_product_lookup_df, rec_product_lookup_path, rec_lookup_err = safe_read_table(
    csv_path=registry["rec_product_lookup_csv"], preferred_label="rec_product_lookup_csv"
)
known_users_df, known_users_path, known_users_err = safe_read_table(
    csv_path=registry["known_users_csv"], preferred_label="known_users_csv"
)
known_items_df, known_items_path, known_items_err = safe_read_table(
    csv_path=registry["known_items_csv"], preferred_label="known_items_csv"
)
candidate_items_df, candidate_items_path, candidate_items_err = safe_read_table(
    csv_path=registry["candidate_items_csv"], preferred_label="candidate_items_csv"
)
popularity_fallback_df, popularity_fallback_path, popularity_fallback_err = safe_read_table(
    csv_path=registry["popular_products_fallback_csv"], preferred_label="popular_products_fallback_csv"
)

frequent_itemsets_df, frequent_itemsets_path, fi_err = safe_read_table(
    csv_path=registry["frequent_itemsets_csv"], preferred_label="frequent_itemsets_csv"
)
association_rules_df, association_rules_path, ar_err = safe_read_table(
    csv_path=registry["association_rules_csv"], preferred_label="association_rules_csv"
)
fpgrowth_grid_df, fpgrowth_grid_path, fg_grid_err = safe_read_table(
    csv_path=registry["fpgrowth_threshold_search_csv"], preferred_label="fpgrowth_threshold_search_csv"
)
fpgrowth_top_rules_preview_df, fpgrowth_top_rules_preview_path, top_rules_preview_err = safe_read_table(
    csv_path=registry["top_rules_preview_csv"], preferred_label="top_rules_preview_csv"
)
fpgrowth_top_rules_report_df, fpgrowth_top_rules_report_path, top_rules_report_err = safe_read_table(
    csv_path=registry["top_rules_report_csv"], preferred_label="top_rules_report_csv"
)
fpgrowth_top_itemsets_preview_df, fpgrowth_top_itemsets_preview_path, top_itemsets_preview_err = safe_read_table(
    csv_path=registry["top_itemsets_preview_csv"], preferred_label="top_itemsets_preview_csv"
)

artifact_audit_items = [
    ("classification", "best_classifier_model", classifier_model, classifier_model_path, classifier_err, "core"),
    ("classification", "best_text_classifier_model", text_classifier_model, text_classifier_model_path, text_classifier_err, "support"),
    ("regression", "best_regressor_model", regressor_model, regressor_model_path, regressor_err, "core"),
    ("clustering", "kmeans_model", kmeans_model, kmeans_model_path, kmeans_err, "support"),
    ("clustering", "gmm_model", gmm_model, gmm_model_path, gmm_err, "support"),
    ("clustering", "rfm_scaler", rfm_scaler, rfm_scaler_path, rfm_scaler_err, "core"),
    ("clustering", "rfm_pca", rfm_pca, rfm_pca_path, rfm_pca_err, "support"),
    ("recommendation", "best_surprise_model", surprise_model, surprise_model_path, surprise_err, "core"),
    ("recommendation", "seen_items_map", seen_items_map, seen_items_map_path, seen_items_map_err, "core"),
    ("recommendation", "surprise_deployment_bundle", surprise_bundle, surprise_bundle_path, surprise_bundle_err, "core"),
    ("recommendation", "item_knn_neighbors_model", item_knn_model, item_knn_model_path, item_knn_err, "support"),
    ("recommendation", "rec_product_lookup_df", rec_product_lookup_df, rec_product_lookup_path, rec_lookup_err, "support"),
    ("recommendation", "known_users_df", known_users_df, known_users_path, known_users_err, "support"),
    ("recommendation", "known_items_df", known_items_df, known_items_path, known_items_err, "support"),
    ("recommendation", "candidate_items_df", candidate_items_df, candidate_items_path, candidate_items_err, "support"),
    ("recommendation", "popularity_fallback_df", popularity_fallback_df, popularity_fallback_path, popularity_fallback_err, "support"),
    ("fpgrowth", "frequent_itemsets_df", frequent_itemsets_df, frequent_itemsets_path, fi_err, "core"),
    ("fpgrowth", "association_rules_df", association_rules_df, association_rules_path, ar_err, "core"),
    ("fpgrowth", "fpgrowth_grid_df", fpgrowth_grid_df, fpgrowth_grid_path, fg_grid_err, "support"),
    ("fpgrowth", "fpgrowth_top_rules_preview_df", fpgrowth_top_rules_preview_df, fpgrowth_top_rules_preview_path, top_rules_preview_err, "support"),
    ("fpgrowth", "fpgrowth_top_rules_report_df", fpgrowth_top_rules_report_df, fpgrowth_top_rules_report_path, top_rules_report_err, "support"),
    ("fpgrowth", "fpgrowth_top_itemsets_preview_df", fpgrowth_top_itemsets_preview_df, fpgrowth_top_itemsets_preview_path, top_itemsets_preview_err, "support"),
]

for module, object_name, obj, obj_path, obj_err, severity in artifact_audit_items:
    loaded = obj is not None
    obj_type = type(obj).__name__ if loaded else None
    notes = str(obj_path) if obj_err is None else f"{obj_err} | path={obj_path}"
    add_object_audit(module, object_name, loaded, object_type=obj_type, notes=notes)
    add_check(module, f"{object_name}_loaded", loaded, notes=notes, severity=severity)

object_audit_df = pd.DataFrame(object_audit_rows)
display(Markdown("### Object audit"))
display(object_audit_df)

object_audit_output_path = METRIC_DIR / "integration_object_audit.csv"
object_audit_df.to_csv(object_audit_output_path, index=False)
print("Đã lưu:", object_audit_output_path)


### Object audit

,module,object_name,loaded,object_type,notes
0,classification,best_classifier_model,True,Pipeline,C:\Users\trant\OneDrive\Tài liệu\Documents\U...
1,classification,best_text_classifier_model,True,Pipeline,C:\Users\trant\OneDrive\Tài liệu\Documents\U...
2,regression,best_regressor_model,True,TransformedTargetRegressor,C:\Users\trant\OneDrive\Tài liệu\Documents\U...
3,clustering,kmeans_model,True,KMeans,C:\Users\trant\OneDrive\Tài liệu\Documents\U...
4,clustering,gmm_model,True,GaussianMixture,C:\Users\trant\OneDrive\Tài liệu\Documents\U...
5,clustering,rfm_scaler,True,StandardScaler,C:\Users\trant\OneDrive\Tài liệu\Documents\U...
6,clustering,rfm_pca,True,PCA,C:\Users\trant\OneDrive\Tài liệu\Documents\U...
7,recommendation,best_surprise_model,True,SVD,C:\Users\trant\OneDrive\Tài liệu\Documents\U...
8,recommendation,seen_items_map,True,dict,C:\Users\trant\OneDrive\Tài liệu\Documents\U...
9,recommendation,surprise_deployment_bundle,True,dict,C:\Users\trant\OneDrive\Tài liệu\Documents\U...


Đã lưu: C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project\artifacts\metrics\integration_object_audit.csv


## Cell 8: Trích xuất schema đầu vào cho từng module

**Mục tiêu**
- Xác định schema tối thiểu cần thiết để inference cho:
  - classification
  - regression
  - clustering

**Cách làm**
- Ưu tiên suy schema trực tiếp từ `preprocessor.transformers_` / `transformers`
- Nếu không có thì fallback sang:
  - `feature_names_in_`
  - danh sách feature từ summary JSON
  - danh sách feature mặc định đã khóa

**Vì sao cell này quan trọng**
- UI / API có thể đưa vào bảng dữ liệu nhiều cột hơn mức tối thiểu.
- Điều cần kiểm tra là:
  - **không thiếu cột bắt buộc**,
  - không phải là “chỉ có đúng số cột”.

**Đầu ra mong đợi**
- `schema_audit_df`
- `integration_schema_audit.csv`
- bảng tóm tắt feature schema cho 3 module


In [8]:
# =========================
# SCHEMA EXTRACTION HELPERS
# =========================
def flatten_transformer_columns(transformers):
    cols = []
    for _, _, part_cols in transformers:
        if part_cols is None or part_cols == "drop":
            continue
        if isinstance(part_cols, slice):
            continue
        if isinstance(part_cols, (list, tuple, pd.Index, np.ndarray)):
            cols.extend(list(part_cols))
        else:
            cols.append(part_cols)
    return unique_preserve_order(cols)

def try_get_pipeline_from_estimator(estimator):
    if estimator is None:
        return None

    # sklearn Pipeline
    if hasattr(estimator, "named_steps"):
        return estimator

    # TransformedTargetRegressor
    if hasattr(estimator, "regressor_") and hasattr(estimator.regressor_, "named_steps"):
        return estimator.regressor_
    if hasattr(estimator, "regressor") and hasattr(estimator.regressor, "named_steps"):
        return estimator.regressor

    return None

def extract_input_feature_columns(estimator):
    """
    Ưu tiên lấy schema từ preprocessor.transformers_ / transformers.
    Nếu không có, fallback sang feature_names_in_ nếu tồn tại.
    """
    if estimator is None:
        return []

    pipe = try_get_pipeline_from_estimator(estimator)
    if pipe is not None:
        pre = pipe.named_steps.get("preprocessor")
        if pre is not None:
            transformers = getattr(pre, "transformers_", None)
            if transformers is None:
                transformers = getattr(pre, "transformers", [])
            if transformers is not None:
                cols = flatten_transformer_columns(transformers)
                if len(cols) > 0:
                    return cols

    if hasattr(estimator, "feature_names_in_"):
        return list(estimator.feature_names_in_)

    # TTR fitted sometimes exposes feature_names_in_
    if hasattr(estimator, "regressor_") and hasattr(estimator.regressor_, "feature_names_in_"):
        return list(estimator.regressor_.feature_names_in_)

    return []

def infer_categorical_columns_from_sample(df, feature_cols):
    if df is None:
        return []
    cats = []
    for col in feature_cols:
        if col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
                cats.append(col)
    return cats

# -------------------------
# CLASSIFICATION SCHEMA
# -------------------------
classification_feature_cols = extract_input_feature_columns(classifier_model)
classification_feature_cols = classification_feature_cols or [
    "item_count", "unique_products", "unique_sellers", "price_sum", "freight_value_sum",
    "price_mean", "payment_value_sum", "payment_installments_max", "basket_value",
    "purchase_year", "purchase_month", "purchase_day", "purchase_hour", "purchase_dayofweek",
    "customer_state", "main_category", "payment_type_mode"
]
classification_categorical_cols = infer_categorical_columns_from_sample(orders_base_final, classification_feature_cols)
classification_numeric_cols = [c for c in classification_feature_cols if c not in classification_categorical_cols]

# -------------------------
# REGRESSION SCHEMA
# -------------------------
regression_feature_cols = []
if isinstance(regression_summary, dict) and "_error" not in regression_summary:
    regression_feature_cols = (regression_summary.get("numeric_features", []) or []) + (regression_summary.get("categorical_features", []) or [])

if len(regression_feature_cols) == 0:
    regression_feature_cols = extract_input_feature_columns(regressor_model)

regression_feature_cols = regression_feature_cols or [
    "item_count", "unique_products", "unique_sellers", "price_mean", "payment_installments_max",
    "purchase_year", "purchase_month", "purchase_day", "purchase_hour", "purchase_dayofweek",
    "customer_state", "main_category", "payment_type_mode"
]
regression_categorical_cols = infer_categorical_columns_from_sample(orders_base_final, regression_feature_cols)
regression_numeric_cols = [c for c in regression_feature_cols if c not in regression_categorical_cols]

# -------------------------
# CLUSTERING SCHEMA
# -------------------------
RFM_ID_COL = "customer_unique_id"
RFM_FEATURES = ["recency_days", "frequency_orders", "monetary_value"]
use_log1p_transform = True
if isinstance(clustering_summary, dict) and "_error" not in clustering_summary:
    use_log1p_transform = bool(clustering_summary.get("use_log1p_transform", True))

# -------------------------
# SCHEMA AUDIT AGAINST TABLES
# -------------------------
if orders_base_final is not None:
    add_schema_audit("classification", "classification_input_schema", classification_feature_cols, list(orders_base_final.columns))
    add_schema_audit("regression", "regression_input_schema", regression_feature_cols, list(orders_base_final.columns))

if rfm_df is not None:
    add_schema_audit("clustering", "rfm_input_schema", [RFM_ID_COL] + RFM_FEATURES, list(rfm_df.columns))

schema_audit_df = pd.DataFrame(schema_audit_rows)
display(Markdown("### Schema audit"))
display(schema_audit_df)

schema_audit_output_path = METRIC_DIR / "integration_schema_audit.csv"
schema_audit_df.to_csv(schema_audit_output_path, index=False)
print("Đã lưu:", schema_audit_output_path)

display(Markdown("### Feature schema tóm tắt"))
display(pd.DataFrame([
    {
        "module": "classification",
        "n_features": len(classification_feature_cols),
        "numeric_features": ", ".join(classification_numeric_cols),
        "categorical_features": ", ".join(classification_categorical_cols),
    },
    {
        "module": "regression",
        "n_features": len(regression_feature_cols),
        "numeric_features": ", ".join(regression_numeric_cols),
        "categorical_features": ", ".join(regression_categorical_cols),
    },
    {
        "module": "clustering",
        "n_features": len(RFM_FEATURES),
        "numeric_features": ", ".join(RFM_FEATURES),
        "categorical_features": "",
    },
]))

### Schema audit

,module,schema_name,n_expected,n_provided,n_missing,n_extra,missing_columns,extra_columns
0,classification,classification_input_schema,17,52,0,35,,"order_id, customer_id, order_status, order_pur..."
1,regression,regression_input_schema,13,52,0,39,,"order_id, customer_id, order_status, order_pur..."
2,clustering,rfm_input_schema,4,4,0,0,,


Đã lưu: C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project\artifacts\metrics\integration_schema_audit.csv


### Feature schema tóm tắt

,module,n_features,numeric_features,categorical_features
0,classification,17,"item_count, unique_products, unique_sellers, p...","customer_state, main_category, payment_type_mode"
1,regression,13,"item_count, unique_products, unique_sellers, p...","customer_state, main_category, payment_type_mode"
2,clustering,3,"recency_days, frequency_orders, monetary_value",


## Cell 9: Chạy smoke test cho module classification

**Mục tiêu**
- Kiểm tra inference thật cho:
  - classifier tabular
  - classifier text

**Đầu vào**
- `orders_base_final` từ notebook 02
- `best_classifier_model` và `best_text_classifier_model` từ notebook 03

**Logic thực hiện**
1. Tạo sample tabular tốt nhất theo mức độ đầy đủ dữ liệu.
2. Fill tối thiểu để tránh lỗi do null ở lúc predict.
3. Gọi `.predict()` / `.predict_proba()` cho nhánh tabular.
4. Tạo 3 câu text mẫu:
   - tích cực
   - tiêu cực
   - chuỗi rỗng
5. Gọi model text để kiểm tra edge case.

**Đầu ra mong đợi**
- payload classification tabular
- payload classification text
- check pass/fail cho known sample và text inference


In [9]:
# =========================
# CLASSIFICATION SMOKE TEST
# =========================
classification_sample = None
classification_payload = {}
classification_edge_payload = {}
text_classification_payload = {}

def build_sample_from_df(df, feature_cols):
    if df is None or len(df) == 0:
        return None

    available_cols = [c for c in feature_cols if c in df.columns]
    if len(available_cols) == 0:
        return None

    sample_df = df[available_cols].copy()

    # ưu tiên chọn dòng có nhiều non-null nhất
    completeness = sample_df.notna().sum(axis=1)
    best_idx = completeness.sort_values(ascending=False).index[0]

    row = sample_df.loc[[best_idx]].copy()

    # fill tối thiểu để tránh vỡ predict
    for col in row.columns:
        if row[col].isna().any():
            if pd.api.types.is_numeric_dtype(sample_df[col]):
                fill_val = sample_df[col].median()
                if pd.isna(fill_val):
                    fill_val = 0
            else:
                fill_val = first_non_null_value(sample_df[col].astype(str), default="unknown")
            row[col] = row[col].fillna(fill_val)

    return row

def get_classifier_prediction_payload(model, X):
    pred = model.predict(X)
    pred_val = pred[0]

    payload = {
        "predicted_label": int(pred_val) if isinstance(pred_val, (np.integer, int)) else str(pred_val),
        "confidence": None,
        "class_probabilities": None,
        "raw_score": None
    }

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)[0]
        classes = list(model.classes_)
        prob_map = {str(cls): float(prob) for cls, prob in zip(classes, proba)}
        pred_idx = classes.index(pred_val)
        payload["confidence"] = float(proba[pred_idx])
        payload["class_probabilities"] = prob_map

    elif hasattr(model, "decision_function"):
        score = model.decision_function(X)
        score = np.ravel(score)
        if len(score) == 1:
            payload["raw_score"] = float(score[0])
            payload["confidence"] = float(abs(score[0]))
        else:
            payload["raw_score"] = [float(s) for s in score]

    return payload

if classifier_model is not None and orders_base_final is not None:
    classification_sample = build_sample_from_df(orders_base_final, classification_feature_cols)

    try:
        classification_payload = get_classifier_prediction_payload(classifier_model, classification_sample)
        ui_payload_preview["classification_tabular"] = make_json_serializable(classification_payload)
        add_check("classification", "tabular_predict_sample", True, notes=str(classification_payload))
    except Exception as e:
        add_check("classification", "tabular_predict_sample", False, notes=str(e))

    # edge case: category lạ nhưng schema vẫn đúng
    try:
        edge_sample = classification_sample.copy()
        for col in classification_categorical_cols:
            if col in edge_sample.columns:
                edge_sample[col] = f"unseen_{col}"
        classification_edge_payload = get_classifier_prediction_payload(classifier_model, edge_sample)
        ui_payload_preview["classification_tabular_unknown_category"] = make_json_serializable(classification_edge_payload)
        add_check("classification", "tabular_predict_unknown_categories", True, notes=str(classification_edge_payload))
    except Exception as e:
        add_check("classification", "tabular_predict_unknown_categories", False, notes=str(e))

display(Markdown("### Classification sample input"))
display(classification_sample if classification_sample is not None else pd.DataFrame({"message": ["Không tạo được classification sample."]}))

display(Markdown("### Classification payload"))
display(pd.DataFrame([classification_payload]) if classification_payload else pd.DataFrame({"message": ["Classification payload rỗng hoặc predict thất bại."]}))

# -------------------------
# TEXT CLASSIFIER (SUPPLEMENTAL)
# -------------------------
if text_classifier_model is not None:
    text_samples = [
        "produto excelente chegou rápido e bem embalado",
        "entrega atrasou muito e o produto veio com defeito",
        ""
    ]
    text_rows = []
    for txt in text_samples:
        try:
            pred = text_classifier_model.predict([txt])[0]
            row = {"text": txt, "predicted_label": int(pred) if isinstance(pred, (np.integer, int)) else str(pred)}
            if hasattr(text_classifier_model, "decision_function"):
                score = text_classifier_model.decision_function([txt])
                row["raw_score"] = float(np.ravel(score)[0])
            text_rows.append(row)
        except Exception as e:
            text_rows.append({"text": txt, "predicted_label": "ERROR", "raw_score": str(e)})

    text_classifier_result_df = pd.DataFrame(text_rows)
    text_classification_payload = text_rows
    ui_payload_preview["classification_text"] = make_json_serializable(text_rows)
    add_check(
        "classification",
        "text_classifier_smoke_test",
        bool((text_classifier_result_df["predicted_label"] != "ERROR").all()),
        notes=f"rows={len(text_classifier_result_df)}"
    )

    display(Markdown("### Text classifier smoke test"))
    display(text_classifier_result_df)

### Classification sample input

,item_count,unique_products,unique_sellers,price_sum,freight_value_sum,price_mean,payment_value_sum,payment_installments_max,basket_value,purchase_year,purchase_month,purchase_day,purchase_hour,purchase_dayofweek,customer_state,main_category,payment_type_mode
0,1.0,1.0,1.0,29.99,8.72,29.99,38.71,1.0,38.71,2017,10,2,10,0,SP,housewares,voucher


### Classification payload

,predicted_label,confidence,class_probabilities,raw_score
0,1,0.606725,"{'0': 0.3932750024994728, '1': 0.6067249975005...",None


### Text classifier smoke test

,text,predicted_label,raw_score
0,produto excelente chegou rápido e bem embalado,1,4.825468
1,entrega atrasou muito e o produto veio com def...,0,-2.408754
2,,1,0.557157


## Cell 10: Chạy smoke test cho module regression

**Mục tiêu**
- Kiểm tra model regression có load và predict được ở mức inference.

**Đầu vào**
- `orders_base_final`
- `best_regressor_model`

**Logic thực hiện**
- dựng 1 sample inference từ processed table
- chạy `.predict()`
- nếu có `payment_value_sum` ở dòng tham chiếu thì ghi thêm:
  - `y_true_reference`
  - `absolute_error_vs_reference`

**Lưu ý quan trọng**
- Cell này **không dùng để chấm lại chất lượng mô hình**.
- Nó chỉ dùng để chứng minh nhánh regression **suy luận được** và payload số học không lỗi.

**Đầu ra mong đợi**
- payload regression
- payload regression edge-case
- check `predict_regression_sample`


In [10]:
# =========================
# REGRESSION SMOKE TEST
# =========================
regression_sample = None
regression_payload = {}
regression_edge_payload = {}

if regressor_model is not None and orders_base_final is not None:
    regression_sample = build_sample_from_df(orders_base_final, regression_feature_cols)

    try:
        y_pred = float(np.ravel(regressor_model.predict(regression_sample))[0])
        y_true = None
        if "payment_value_sum" in orders_base_final.columns and regression_sample.index[0] in orders_base_final.index:
            y_true = float(orders_base_final.loc[regression_sample.index[0], "payment_value_sum"])

        regression_payload = {
            "predicted_payment_value_sum": y_pred,
            "y_true_reference": y_true,
            "absolute_error_vs_reference": None if y_true is None else float(abs(y_true - y_pred))
        }
        ui_payload_preview["regression"] = make_json_serializable(regression_payload)
        add_check("regression", "predict_sample", np.isfinite(y_pred) and y_pred >= 0, notes=str(regression_payload))
    except Exception as e:
        add_check("regression", "predict_sample", False, notes=str(e))

    try:
        edge_sample = regression_sample.copy()
        for col in regression_categorical_cols:
            if col in edge_sample.columns:
                edge_sample[col] = f"unseen_{col}"
        for col in regression_numeric_cols:
            if col in edge_sample.columns and pd.api.types.is_numeric_dtype(edge_sample[col]):
                edge_sample[col] = float(edge_sample[col].iloc[0]) if not pd.isna(edge_sample[col].iloc[0]) else 0.0

        y_edge = float(np.ravel(regressor_model.predict(edge_sample))[0])
        regression_edge_payload = {"predicted_payment_value_sum": y_edge}
        ui_payload_preview["regression_unknown_category"] = make_json_serializable(regression_edge_payload)
        add_check("regression", "predict_unknown_categories", np.isfinite(y_edge) and y_edge >= 0, notes=str(regression_edge_payload))
    except Exception as e:
        add_check("regression", "predict_unknown_categories", False, notes=str(e))

display(Markdown("### Regression sample input"))
display(regression_sample if regression_sample is not None else pd.DataFrame({"message": ["Không tạo được regression sample."]}))

display(Markdown("### Regression payload"))
display(pd.DataFrame([regression_payload]) if regression_payload else pd.DataFrame({"message": ["Regression payload rỗng hoặc predict thất bại."]}))

### Regression sample input

,item_count,unique_products,unique_sellers,price_mean,payment_installments_max,purchase_year,purchase_month,purchase_day,purchase_hour,purchase_dayofweek,customer_state,main_category,payment_type_mode
0,1.0,1.0,1.0,29.99,1.0,2017,10,2,10,0,SP,housewares,voucher


### Regression payload

,predicted_payment_value_sum,y_true_reference,absolute_error_vs_reference
0,40.664622,38.71,1.954622


## Cell 11: Chọn mô hình clustering để suy luận và kiểm tra phân cụm

**Mục tiêu**
- Chọn model clustering inference chính cho notebook 08.
- Tạo payload phân cụm dùng được cho UI.

**Logic chọn model**
- Mặc định ép dùng `kmeans` nếu model tồn tại (`FORCE_CLUSTER_MODEL_NAME = "kmeans"`).
- Nếu không ép, notebook có thể so theo summary để chọn giữa `kmeans` và `gmm`.

**Đầu vào**
- `rfm_df`
- `rfm_scaler`
- `kmeans_model` / `gmm_model`
- cluster profile CSV từ notebook 05

**Đầu ra mong đợi**
- payload clustering cho 1 khách hàng thật
- payload clustering edge-case
- map được `assigned_cluster -> segment_name`


In [11]:
# =========================
# CLUSTERING MODEL SELECTION FOR INFERENCE
# =========================
best_cluster_model_name = None
best_cluster_model = None
best_cluster_profile_df = None
best_cluster_label_col = None
FORCE_CLUSTER_MODEL_NAME = "kmeans"   # "kmeans", "gmm", hoặc None
kmeans_sil = None
gmm_sil = None

if isinstance(clustering_summary, dict) and "_error" not in clustering_summary:
    kmeans_sil = clustering_summary.get("kmeans_best_silhouette", None)
    gmm_sil = clustering_summary.get("gmm_best_silhouette", None)

if FORCE_CLUSTER_MODEL_NAME == "kmeans" and kmeans_model is not None:
    best_cluster_model_name = "kmeans"
    best_cluster_model = kmeans_model
elif FORCE_CLUSTER_MODEL_NAME == "gmm" and gmm_model is not None:
    best_cluster_model_name = "gmm"
    best_cluster_model = gmm_model
else:
    # logic cũ
    if kmeans_model is not None and gmm_model is not None:
        if gmm_sil is not None and kmeans_sil is not None and gmm_sil > kmeans_sil:
            best_cluster_model_name = "gmm"
            best_cluster_model = gmm_model
        else:
            best_cluster_model_name = "kmeans"
            best_cluster_model = kmeans_model
    elif kmeans_model is not None:
        best_cluster_model_name = "kmeans"
        best_cluster_model = kmeans_model
    elif gmm_model is not None:
        best_cluster_model_name = "gmm"
        best_cluster_model = gmm_model

kmeans_profile_df, _, _ = safe_read_table(csv_path=registry["kmeans_profile_csv"], preferred_label="kmeans_profile_csv")
gmm_profile_df, _, _ = safe_read_table(csv_path=registry["gmm_profile_csv"], preferred_label="gmm_profile_csv")

if best_cluster_model_name == "kmeans":
    best_cluster_profile_df = kmeans_profile_df
    best_cluster_label_col = "cluster_kmeans"
elif best_cluster_model_name == "gmm":
    best_cluster_profile_df = gmm_profile_df
    best_cluster_label_col = "cluster_gmm"

clustering_payload = {}
clustering_edge_payload = {}

def map_cluster_to_segment(cluster_id, profile_df, label_col):
    if profile_df is None or label_col is None or label_col not in profile_df.columns:
        return f"Cluster {cluster_id}"

    subset = profile_df[profile_df[label_col].astype(str) == str(cluster_id)]
    if len(subset) == 0:
        subset = profile_df[profile_df[label_col] == cluster_id]

    if len(subset) == 0:
        return f"Cluster {cluster_id}"

    row = subset.iloc[0]
    if "segment_name" in row and pd.notna(row["segment_name"]):
        return str(row["segment_name"])

    return f"Cluster {cluster_id}"

if best_cluster_model is not None and rfm_scaler is not None and rfm_df is not None:
    # known sample
    clustering_sample = rfm_df[[RFM_ID_COL] + RFM_FEATURES].dropna().iloc[[0]].copy()
    X_cluster = clustering_sample[RFM_FEATURES].astype(float).copy()

    if use_log1p_transform:
        X_cluster = np.log1p(X_cluster)

    X_scaled = rfm_scaler.transform(X_cluster)
    pred_cluster = best_cluster_model.predict(X_scaled)
    pred_cluster_val = int(np.ravel(pred_cluster)[0])

    segment_name = map_cluster_to_segment(pred_cluster_val, best_cluster_profile_df, best_cluster_label_col)

    clustering_payload = {
        "cluster_model_used": best_cluster_model_name,
        "customer_unique_id": str(clustering_sample.iloc[0][RFM_ID_COL]),
        "assigned_cluster": pred_cluster_val,
        "segment_name": segment_name,
        "use_log1p_transform": bool(use_log1p_transform)
    }
    ui_payload_preview["clustering"] = make_json_serializable(clustering_payload)
    add_check("clustering", "predict_known_rfm_sample", True, notes=str(clustering_payload))

    # edge sample
    edge_row = pd.DataFrame([{
        "recency_days": float(rfm_df["recency_days"].max()),
        "frequency_orders": max(1.0, float(rfm_df["frequency_orders"].min())),
        "monetary_value": max(0.0, float(rfm_df["monetary_value"].median()))
    }])

    X_edge = edge_row.copy()
    if use_log1p_transform:
        X_edge = np.log1p(X_edge)

    X_edge_scaled = rfm_scaler.transform(X_edge)
    edge_cluster = int(np.ravel(best_cluster_model.predict(X_edge_scaled))[0])

    clustering_edge_payload = {
        "cluster_model_used": best_cluster_model_name,
        "assigned_cluster": edge_cluster,
        "segment_name": map_cluster_to_segment(edge_cluster, best_cluster_profile_df, best_cluster_label_col)
    }
    ui_payload_preview["clustering_edge_case"] = make_json_serializable(clustering_edge_payload)
    add_check("clustering", "predict_edge_rfm_sample", True, notes=str(clustering_edge_payload))
else:
    add_check(
        "clustering",
        "predict_known_rfm_sample",
        False,
        notes=f"best_cluster_model={best_cluster_model_name}, scaler_loaded={rfm_scaler is not None}, rfm_df_loaded={rfm_df is not None}"
    )

display(Markdown("### Clustering payload"))
display(pd.DataFrame([clustering_payload]) if clustering_payload else pd.DataFrame({"message": ["Chưa tạo được clustering payload."]}))

display(Markdown("### Clustering edge payload"))
display(pd.DataFrame([clustering_edge_payload]) if clustering_edge_payload else pd.DataFrame({"message": ["Chưa tạo được clustering edge payload."]}))

### Clustering payload

,cluster_model_used,customer_unique_id,assigned_cluster,segment_name,use_log1p_transform
0,kmeans,0000366f3b9a7992bf8c76cfdf3221e2,0,High-value lapsed,True


### Clustering edge payload

,cluster_model_used,assigned_cluster,segment_name
0,kmeans,1,Low-value lapsed


## Cell 12: Định nghĩa các hàm helper cho recommendation

**Mục tiêu**
- Tạo lớp helper đủ dùng để:
  - chuẩn hóa `product_lookup`
  - enrich metadata sản phẩm
  - build fallback output
  - recommend top-N cho user
  - recommend neighbor cho `product_id`

**Vì sao cell này quan trọng**
- Recommendation là nhánh có nhiều edge case:
  - unknown user
  - no unseen items
  - unknown item
  - thiếu metadata sản phẩm
- Nếu helper không đủ chặt, payload UI rất dễ lỗi hoặc nghèo thông tin.

**Đầu ra mong đợi**
- Các helper đảm bảo:
  - output có metadata sản phẩm tối thiểu,
  - không để lại cột `_x/_y`,
  - fallback hoạt động rõ ràng và giải thích được.


In [12]:
# =========================
# RECOMMENDATION HELPERS
# =========================
LOOKUP_META_COLS = [
    "product_category_name_english",
    "avg_price",
    "purchase_count",
    "rating_mean",
    "rating_count",
    "weighted_rating"
]

def finalize_product_lookup(df):
    if df is None or len(df) == 0:
        return df

    out = df.copy()

    if "product_id" in out.columns:
        out["product_id"] = out["product_id"].astype(str).str.strip()
        out = out.dropna(subset=["product_id"]).drop_duplicates("product_id")

    for col in ["avg_price", "purchase_count", "rating_mean", "rating_count", "weighted_rating"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    if "product_category_name_english" not in out.columns:
        out["product_category_name_english"] = "unknown"

    out["product_category_name_english"] = (
        out["product_category_name_english"]
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .replace("", "unknown")
    )

    preferred_cols = ["product_id", "product_category_name_english", "avg_price", "purchase_count", "rating_mean", "rating_count", "weighted_rating"]
    existing_cols = [c for c in preferred_cols if c in out.columns]
    other_cols = [c for c in out.columns if c not in existing_cols]

    out = out[existing_cols + other_cols].copy()
    return out

def coalesce_lookup_columns(df):
    if df is None or len(df) == 0:
        return df

    out = df.copy()

    for col in LOOKUP_META_COLS:
        x_col = f"{col}_x"
        y_col = f"{col}_y"

        if col not in out.columns:
            if x_col in out.columns and y_col in out.columns:
                out[col] = out[x_col].combine_first(out[y_col])
            elif x_col in out.columns:
                out[col] = out[x_col]
            elif y_col in out.columns:
                out[col] = out[y_col]

        drop_cols = [c for c in [x_col, y_col] if c in out.columns]
        if drop_cols:
            out = out.drop(columns=drop_cols)

    if "product_category_name_english" in out.columns:
        out["product_category_name_english"] = (
            out["product_category_name_english"]
            .fillna("unknown")
            .astype(str)
            .str.strip()
            .replace("", "unknown")
        )

    return out

def enrich_with_product_lookup(df, product_lookup_df):
    if df is None or len(df) == 0:
        return df

    out = coalesce_lookup_columns(df)

    if any(col in out.columns for col in LOOKUP_META_COLS):
        return out

    if product_lookup_df is None or len(product_lookup_df) == 0:
        return out

    lookup = finalize_product_lookup(product_lookup_df)

    preferred_lookup_cols = ["product_id"] + [c for c in LOOKUP_META_COLS if c in lookup.columns]
    lookup = lookup[preferred_lookup_cols].copy()

    if "product_id" in out.columns:
        out["product_id"] = out["product_id"].astype(str).str.strip()

    enriched = out.merge(lookup, on="product_id", how="left")
    return coalesce_lookup_columns(enriched)

def build_fallback_output(fallback_df, n=10, reason="cold_start_popularity"):
    if fallback_df is None or len(fallback_df) == 0:
        return pd.DataFrame(columns=["product_id", "estimated_score", "reason"])

    out = fallback_df.copy()
    if "product_id" in out.columns:
        out["product_id"] = out["product_id"].astype(str).str.strip()

    out = out.head(n).copy()
    out["estimated_score"] = out.get("weighted_rating", np.nan)
    out["reason"] = reason
    return coalesce_lookup_columns(out)

def recommend_top_n_for_user(raw_uid, algo, all_item_ids, seen_map, product_lookup_df, fallback_df, known_users, n=10):
    raw_uid = str(raw_uid)

    if raw_uid not in known_users:
        return build_fallback_output(fallback_df, n=n, reason="cold_start_popularity")

    seen_items = seen_map.get(raw_uid, set())
    candidates = [iid for iid in all_item_ids if iid not in seen_items]

    if len(candidates) == 0:
        return build_fallback_output(fallback_df, n=n, reason="no_unseen_items_fallback")

    scored = []
    for iid in candidates:
        try:
            est = algo.predict(raw_uid, iid).est
        except Exception:
            est = np.nan
        scored.append((iid, est))

    reco_df = pd.DataFrame(scored, columns=["product_id", "estimated_score"])
    reco_df = reco_df.dropna(subset=["estimated_score"])

    if reco_df.empty:
        return build_fallback_output(fallback_df, n=n, reason="prediction_failed_fallback")

    reco_df = (
        reco_df
        .sort_values(["estimated_score", "product_id"], ascending=[False, True])
        .head(n)
        .copy()
    )
    reco_df["reason"] = "collaborative_filtering"
    reco_df = enrich_with_product_lookup(reco_df, product_lookup_df)
    reco_df = coalesce_lookup_columns(reco_df)
    return reco_df

def build_neighbor_fallback(fallback_df, query_product_id, n=10, reason="unknown_item_fallback"):
    out = build_fallback_output(fallback_df, n=n + 5, reason=reason).copy()
    if "product_id" in out.columns:
        out = out[out["product_id"].astype(str) != str(query_product_id)].copy()
    out = out.head(n).copy()
    out["neighbor_rank"] = range(1, len(out) + 1)
    out["similarity"] = np.nan
    return out

def recommend_similar_products(product_id, knn_algo, product_lookup_df, fallback_df, n=10):
    product_id = str(product_id)

    if knn_algo is None or not hasattr(knn_algo, "trainset"):
        return build_neighbor_fallback(fallback_df, product_id, n=n, reason="item_knn_unavailable_fallback")

    try:
        inner_iid = knn_algo.trainset.to_inner_iid(product_id)
        neighbor_inner_ids = knn_algo.get_neighbors(inner_iid, k=n + 10)

        raw_neighbor_ids = []
        for ni in neighbor_inner_ids:
            raw_iid = knn_algo.trainset.to_raw_iid(ni)
            if raw_iid != product_id and raw_iid not in raw_neighbor_ids:
                raw_neighbor_ids.append(raw_iid)

        raw_neighbor_ids = raw_neighbor_ids[:n]

        pairs = []
        for raw_iid in raw_neighbor_ids:
            sim_val = np.nan
            if hasattr(knn_algo, "sim"):
                try:
                    other_inner_iid = knn_algo.trainset.to_inner_iid(raw_iid)
                    sim_val = float(knn_algo.sim[inner_iid, other_inner_iid])
                except Exception:
                    sim_val = np.nan
            pairs.append((raw_iid, sim_val))

        out = pd.DataFrame(pairs, columns=["product_id", "similarity"])

        if len(out) < n:
            fallback_extra = build_neighbor_fallback(fallback_df, product_id, n=n + 10, reason="fallback_completion").copy()
            used_ids = set(out["product_id"].astype(str).tolist()) | {product_id}
            fallback_extra = fallback_extra[~fallback_extra["product_id"].astype(str).isin(used_ids)].copy()
            fallback_extra = fallback_extra.head(n - len(out)).copy()

            if len(fallback_extra) > 0:
                fallback_extra = fallback_extra[["product_id"]].copy()
                fallback_extra["similarity"] = np.nan
                out = pd.concat([out, fallback_extra], ignore_index=True)

        if out.empty:
            return build_neighbor_fallback(fallback_df, product_id, n=n, reason="unknown_item_fallback")

        out = out.head(n).copy()
        out["neighbor_rank"] = range(1, len(out) + 1)
        out["reason"] = np.where(out["similarity"].notna(), "item_knn_neighbors", "fallback_completion")
        out = enrich_with_product_lookup(out, product_lookup_df)
        out = coalesce_lookup_columns(out)
        return out

    except Exception:
        return build_neighbor_fallback(fallback_df, product_id, n=n, reason="unknown_item_fallback")

## Cell 13: Chạy smoke test cho recommendation

**Mục tiêu**
- Xác minh nhánh recommendation hoạt động ở mức UI payload.

**Các trường hợp kiểm tra**
1. **Known user** → phải tạo được top-N
2. **Unknown user** → phải kích hoạt fallback
3. **Known product** → phải tạo được neighbor list
4. **Unknown product** → phải fallback được

**Đầu vào**
- model / bundle từ notebook 06
- `seen_items_map`
- `known_users.csv`
- `candidate_items.csv`
- `product_lookup`
- popularity fallback

**Đầu ra mong đợi**
- payload cho 4 trường hợp trên
- check pass/fail cho:
  - known_user_top_n
  - unknown_user_fallback
  - known_product_neighbors
  - unknown_product_neighbors_fallback


In [13]:
# =========================
# RECOMMENDATION SMOKE TESTS
# =========================
rec_product_lookup_final = finalize_product_lookup(rec_product_lookup_df if rec_product_lookup_df is not None else processed_product_lookup_df)

known_users_set = set()
candidate_item_ids = []
top_n_default = 10

if known_users_df is not None and "customer_unique_id" in known_users_df.columns:
    known_users_set = set(known_users_df["customer_unique_id"].astype(str).str.strip())

if candidate_items_df is not None and "product_id" in candidate_items_df.columns:
    candidate_item_ids = candidate_items_df["product_id"].astype(str).str.strip().tolist()

if isinstance(surprise_bundle, dict):
    top_n_default = int(surprise_bundle.get("top_n_default", 10) or 10)

recommendation_payload_known_user = {}
recommendation_payload_unknown_user = {}
neighbor_payload_known_item = {}
neighbor_payload_unknown_item = {}

# known user
if surprise_model is not None and isinstance(seen_items_map, dict) and len(candidate_item_ids) > 0 and len(known_users_set) > 0:
    known_user_id = sorted(known_users_set)[0]

    try:
        reco_known_df = recommend_top_n_for_user(
            raw_uid=known_user_id,
            algo=surprise_model,
            all_item_ids=candidate_item_ids,
            seen_map=seen_items_map,
            product_lookup_df=rec_product_lookup_final,
            fallback_df=popularity_fallback_df,
            known_users=known_users_set,
            n=top_n_default
        )
        recommendation_payload_known_user = {
            "customer_unique_id": known_user_id,
            "n_rows": int(len(reco_known_df)),
            "top_5": reco_known_df.head(5).to_dict(orient="records")
        }
        ui_payload_preview["recommendation_known_user"] = make_json_serializable(recommendation_payload_known_user)
        add_check("recommendation", "known_user_top_n", len(reco_known_df) > 0, notes=f"user={known_user_id}, rows={len(reco_known_df)}")
        display(Markdown("### Recommendation - known user"))
        display(reco_known_df.head(10))
    except Exception as e:
        add_check("recommendation", "known_user_top_n", False, notes=str(e))

# unknown user fallback
if surprise_model is not None and isinstance(seen_items_map, dict):
    unknown_user_id = "integration_unknown_user_demo"

    try:
        reco_unknown_df = recommend_top_n_for_user(
            raw_uid=unknown_user_id,
            algo=surprise_model,
            all_item_ids=candidate_item_ids,
            seen_map=seen_items_map,
            product_lookup_df=rec_product_lookup_final,
            fallback_df=popularity_fallback_df,
            known_users=known_users_set,
            n=top_n_default
        )
        recommendation_payload_unknown_user = {
            "customer_unique_id": unknown_user_id,
            "n_rows": int(len(reco_unknown_df)),
            "top_5": reco_unknown_df.head(5).to_dict(orient="records")
        }
        ui_payload_preview["recommendation_unknown_user"] = make_json_serializable(recommendation_payload_unknown_user)

        passed = len(reco_unknown_df) > 0 and ("reason" in reco_unknown_df.columns) and reco_unknown_df["reason"].astype(str).str.contains("fallback|cold_start", case=False, regex=True).any()
        add_check("recommendation", "unknown_user_fallback", passed, notes=f"rows={len(reco_unknown_df)}")
        display(Markdown("### Recommendation - unknown user fallback"))
        display(reco_unknown_df.head(10))
    except Exception as e:
        add_check("recommendation", "unknown_user_fallback", False, notes=str(e))

# known product neighbors
if item_knn_model is not None and len(candidate_item_ids) > 0:
    known_product_id = str(candidate_item_ids[0])

    try:
        neighbor_known_df = recommend_similar_products(
            product_id=known_product_id,
            knn_algo=item_knn_model,
            product_lookup_df=rec_product_lookup_final,
            fallback_df=popularity_fallback_df,
            n=top_n_default
        )
        neighbor_payload_known_item = {
            "query_product_id": known_product_id,
            "n_rows": int(len(neighbor_known_df)),
            "top_5": neighbor_known_df.head(5).to_dict(orient="records")
        }
        ui_payload_preview["recommendation_known_product_neighbors"] = make_json_serializable(neighbor_payload_known_item)

        no_self_neighbor = True
        if "product_id" in neighbor_known_df.columns:
            no_self_neighbor = not neighbor_known_df["product_id"].astype(str).eq(known_product_id).any()

        add_check("recommendation", "known_product_neighbors", len(neighbor_known_df) > 0 and no_self_neighbor, notes=f"rows={len(neighbor_known_df)}, no_self={no_self_neighbor}")
        display(Markdown("### Product neighbors - known item"))
        display(neighbor_known_df.head(10))
    except Exception as e:
        add_check("recommendation", "known_product_neighbors", False, notes=str(e))

# unknown product fallback
if item_knn_model is not None or popularity_fallback_df is not None:
    unknown_product_id = "integration_unknown_product_demo"
    try:
        neighbor_unknown_df = recommend_similar_products(
            product_id=unknown_product_id,
            knn_algo=item_knn_model,
            product_lookup_df=rec_product_lookup_final,
            fallback_df=popularity_fallback_df,
            n=top_n_default
        )
        neighbor_payload_unknown_item = {
            "query_product_id": unknown_product_id,
            "n_rows": int(len(neighbor_unknown_df)),
            "top_5": neighbor_unknown_df.head(5).to_dict(orient="records")
        }
        ui_payload_preview["recommendation_unknown_product_neighbors"] = make_json_serializable(neighbor_payload_unknown_item)

        passed = len(neighbor_unknown_df) > 0 and ("reason" in neighbor_unknown_df.columns)
        add_check("recommendation", "unknown_product_fallback", passed, notes=f"rows={len(neighbor_unknown_df)}")
        display(Markdown("### Product neighbors - unknown item fallback"))
        display(neighbor_unknown_df.head(10))
    except Exception as e:
        add_check("recommendation", "unknown_product_fallback", False, notes=str(e))

### Recommendation - known user

,product_id,estimated_score,reason,product_category_name_english,avg_price,purchase_count,rating_mean,rating_count
0,43b54d1fc56ff394092a3dff6be2d39f,4.169523,collaborative_filtering,health_beauty,110.320000,15,4.800000,15
1,02475368dfb38934fe55f574024fe1d7,4.141379,collaborative_filtering,watches_gifts,29.000000,12,4.272727,11
2,45a9a8115c62ad1ce34845e5ec4cfd48,4.084159,collaborative_filtering,toys,99.990000,5,4.500000,4
3,c7b3cf9de7be95b3e09e7a63315685eb,4.083493,collaborative_filtering,luggage_accessories,82.413158,38,4.542857,35
4,7686fe49c98ae1c537b5cc51992f5d20,4.056411,collaborative_filtering,baby,169.997826,23,4.608696,23
5,82a61259a621866c4ba63743da29a342,4.038303,collaborative_filtering,sports_leisure,124.218750,16,4.733333,15
6,24c66f106f642621e524291a895c9032,4.012121,collaborative_filtering,health_beauty,181.900000,65,4.365079,63
7,b623b7cb05ee3248fbe4a6ecbeed79a4,4.011467,collaborative_filtering,toys,73.129242,66,4.035714,56
8,4d272fe20c57c64dc74b3db191fc31ef,4.008357,collaborative_filtering,bed_bath_table,24.900000,18,4.117647,17
9,cd48f265a63e13b762601f5f794c5fca,3.999320,collaborative_filtering,drinks,49.266129,93,4.273810,84


### Recommendation - unknown user fallback

,product_id,product_category_name_english,rating_mean,rating_count,avg_price,purchase_count,weighted_rating,estimated_score,reason
0,73326828aa5efe1ba096223de496f596,food,4.849057,53,82.611481,54,4.722837,4.722837,cold_start_popularity
1,3e4176d545618ed02f382a3057de32b4,luggage_accessories,4.958333,24,132.379583,24,4.692316,4.692316,cold_start_popularity
2,6a8631b72a2f8729b91514db87e771c0,electronics,4.736842,57,26.641935,62,4.634907,4.634907,cold_start_popularity
3,f7f59e6186e10983a061ac7bdb3494d6,housewares,4.814815,27,34.900000,39,4.609156,4.609156,cold_start_popularity
4,62c89abe1afe3a23c17765d462718a4c,perfumery,4.937500,16,244.812500,16,4.597645,4.597645,cold_start_popularity
5,574597aaf385996112490308e37399ce,housewares,4.826087,23,49.000000,24,4.592084,4.592084,cold_start_popularity
6,2722b7e5f68e776d18fe901638034e54,health_beauty,5.000000,13,28.669231,13,4.588642,4.588642,cold_start_popularity
7,e7f85e7f0203b7b95cc1b4c21b4b070c,cool_stuff,4.850000,20,263.907391,23,4.584625,4.584625,cold_start_popularity
8,475e8a9ddbebf13af503d1c7eccadb1a,office_furniture,4.882353,17,136.163158,19,4.575510,4.575510,cold_start_popularity
9,f8b624d4e475bb8d1bddf1b65c6a64f6,housewares,4.690476,42,179.000000,42,4.568053,4.568053,cold_start_popularity


### Product neighbors - known item

,product_id,similarity,neighbor_rank,reason,product_category_name_english,avg_price,purchase_count,rating_mean,rating_count
0,75c06ee06b201f9b6301d2b5e72993f8,1.0,1,item_knn_neighbors,perfumery,14.581071,28,3.678571,28
1,51250f90d798d377a1928e8a4e2e9ae1,1.0,2,item_knn_neighbors,perfumery,13.990000,12,3.250000,12
2,a6ad77b15e566298a4e8ee2011ab1255,0.0,3,item_knn_neighbors,furniture_decor,29.640000,40,4.210526,19
3,b0961721fd839e9982420e807758a2a6,0.0,4,item_knn_neighbors,garden_tools,57.745669,127,4.270833,96
4,4630761de87581e8b659dc77bb7eb4ee,0.0,5,item_knn_neighbors,luggage_accessories,72.740000,2,5.000000,2
5,6bd248f93425ceeb625a8a97e2404112,0.0,6,item_knn_neighbors,stationery,57.403810,21,4.380952,21
6,57fcacc3434a1f2f2b039c1b4e61f5e1,0.0,7,item_knn_neighbors,telephony,18.900000,11,4.636364,11
7,cf0d29a0ea1464f4c3e5594d0746c51b,0.0,8,item_knn_neighbors,telephony,18.900000,2,5.000000,2
8,189e539d996a9b8ba4bba1a140a024a7,0.0,9,item_knn_neighbors,health_beauty,143.400000,4,3.750000,4
9,a669398f595527fc03acc1ebda6b3cce,0.0,10,item_knn_neighbors,health_beauty,127.500000,2,3.000000,2


### Product neighbors - unknown item fallback

,product_id,product_category_name_english,rating_mean,rating_count,avg_price,purchase_count,weighted_rating,estimated_score,reason,neighbor_rank,similarity
0,73326828aa5efe1ba096223de496f596,food,4.849057,53,82.611481,54,4.722837,4.722837,unknown_item_fallback,1,NaN
1,3e4176d545618ed02f382a3057de32b4,luggage_accessories,4.958333,24,132.379583,24,4.692316,4.692316,unknown_item_fallback,2,NaN
2,6a8631b72a2f8729b91514db87e771c0,electronics,4.736842,57,26.641935,62,4.634907,4.634907,unknown_item_fallback,3,NaN
3,f7f59e6186e10983a061ac7bdb3494d6,housewares,4.814815,27,34.900000,39,4.609156,4.609156,unknown_item_fallback,4,NaN
4,62c89abe1afe3a23c17765d462718a4c,perfumery,4.937500,16,244.812500,16,4.597645,4.597645,unknown_item_fallback,5,NaN
5,574597aaf385996112490308e37399ce,housewares,4.826087,23,49.000000,24,4.592084,4.592084,unknown_item_fallback,6,NaN
6,2722b7e5f68e776d18fe901638034e54,health_beauty,5.000000,13,28.669231,13,4.588642,4.588642,unknown_item_fallback,7,NaN
7,e7f85e7f0203b7b95cc1b4c21b4b070c,cool_stuff,4.850000,20,263.907391,23,4.584625,4.584625,unknown_item_fallback,8,NaN
8,475e8a9ddbebf13af503d1c7eccadb1a,office_furniture,4.882353,17,136.163158,19,4.575510,4.575510,unknown_item_fallback,9,NaN
9,f8b624d4e475bb8d1bddf1b65c6a64f6,housewares,4.690476,42,179.000000,42,4.568053,4.568053,unknown_item_fallback,10,NaN


## Cell 14: Chạy smoke test cho FP-Growth rules

**Mục tiêu**
- Xác minh nhánh FP-Growth đã có đủ artifact để:
  - preview frequent itemsets
  - preview association rules usable cho UI
  - tạo payload đơn giản cho trang “Xu hướng / Association Rules”

**Nguồn dữ liệu ưu tiên**
1. `association_rules.csv` — source chính
2. `top_association_rules_report.csv` — fallback mềm cho báo cáo
3. `top_association_rules_preview.csv` — fallback mềm cho preview
4. `top_frequent_itemsets_preview.csv` — fallback mềm cho itemsets preview

**Nguyên tắc quan trọng**
- Fallback **chỉ dùng để cứu phần preview / payload**, không dùng để “ép PASS giả”.
- Nếu artifact lõi vẫn thiếu, notebook phải tiếp tục ghi nhận fail ở các check core.

**Đầu ra mong đợi**
- schema audit cho itemsets/rules
- payload FP-Growth nếu có thể dựng từ source chính hoặc fallback
- check rõ:
  - artifact lõi có hay không
  - preview có dựng được hay không
  - source nào đang được dùng


In [14]:
# =========================
# FP-GROWTH SMOKE TEST
# =========================
fpgrowth_payload = {}
fpgrowth_rules_preview_df = pd.DataFrame()
fpgrowth_itemsets_preview_df = pd.DataFrame()
fpgrowth_rule_source = "none"
fpgrowth_itemset_source = "none"

required_rule_cols = ["rule_str", "support", "confidence", "lift"]
required_itemset_cols = ["itemsets_str", "support"]

# -------------------------
# SCHEMA AUDIT
# -------------------------
if frequent_itemsets_df is not None:
    available_itemset_cols = list(frequent_itemsets_df.columns)
    add_schema_audit("fpgrowth", "frequent_itemsets_schema", required_itemset_cols, available_itemset_cols)
elif fpgrowth_top_itemsets_preview_df is not None:
    add_schema_audit("fpgrowth", "frequent_itemsets_preview_schema_fallback", required_itemset_cols, list(fpgrowth_top_itemsets_preview_df.columns))

if association_rules_df is not None:
    available_rule_cols = list(association_rules_df.columns)
    add_schema_audit("fpgrowth", "association_rules_schema", required_rule_cols, available_rule_cols)
elif fpgrowth_top_rules_report_df is not None:
    add_schema_audit("fpgrowth", "association_rules_report_schema_fallback", required_rule_cols, list(fpgrowth_top_rules_report_df.columns))
elif fpgrowth_top_rules_preview_df is not None:
    add_schema_audit("fpgrowth", "association_rules_preview_schema_fallback", required_rule_cols, list(fpgrowth_top_rules_preview_df.columns))

# -------------------------
# ITEMSET PREVIEW (MAIN -> FALLBACK)
# -------------------------
if frequent_itemsets_df is not None and len(frequent_itemsets_df) > 0:
    fpgrowth_itemsets_preview_df = frequent_itemsets_df.copy()
    fpgrowth_itemset_source = "frequent_itemsets_csv"
elif fpgrowth_top_itemsets_preview_df is not None and len(fpgrowth_top_itemsets_preview_df) > 0:
    fpgrowth_itemsets_preview_df = fpgrowth_top_itemsets_preview_df.copy()
    fpgrowth_itemset_source = "top_itemsets_preview_csv_fallback"

if len(fpgrowth_itemsets_preview_df) > 0:
    if "support" in fpgrowth_itemsets_preview_df.columns:
        fpgrowth_itemsets_preview_df["support"] = pd.to_numeric(fpgrowth_itemsets_preview_df["support"], errors="coerce")
    if "support_count" in fpgrowth_itemsets_preview_df.columns:
        fpgrowth_itemsets_preview_df["support_count"] = pd.to_numeric(fpgrowth_itemsets_preview_df["support_count"], errors="coerce")
    sort_cols = [c for c in ["support", "support_count"] if c in fpgrowth_itemsets_preview_df.columns]
    if len(sort_cols) > 0:
        ascending = [False for _ in sort_cols]
        fpgrowth_itemsets_preview_df = fpgrowth_itemsets_preview_df.sort_values(sort_cols, ascending=ascending)
    fpgrowth_itemsets_preview_df = fpgrowth_itemsets_preview_df.head(10).copy()

# -------------------------
# RULE PREVIEW (MAIN -> REPORT FALLBACK -> PREVIEW FALLBACK)
# -------------------------
rules_source_df = None
if association_rules_df is not None and len(association_rules_df) > 0:
    rules_source_df = association_rules_df.copy()
    fpgrowth_rule_source = "association_rules_csv"
elif fpgrowth_top_rules_report_df is not None and len(fpgrowth_top_rules_report_df) > 0:
    rules_source_df = fpgrowth_top_rules_report_df.copy()
    fpgrowth_rule_source = "top_rules_report_csv_fallback"
elif fpgrowth_top_rules_preview_df is not None and len(fpgrowth_top_rules_preview_df) > 0:
    rules_source_df = fpgrowth_top_rules_preview_df.copy()
    fpgrowth_rule_source = "top_rules_preview_csv_fallback"

usable_rules_df = pd.DataFrame()

if rules_source_df is not None and len(rules_source_df) > 0:
    fpgrowth_rules_preview_df = rules_source_df.copy()

    for col in ["support", "confidence", "lift", "support_count"]:
        if col in fpgrowth_rules_preview_df.columns:
            fpgrowth_rules_preview_df[col] = pd.to_numeric(fpgrowth_rules_preview_df[col], errors="coerce")

    usable_rules_df = fpgrowth_rules_preview_df.copy()
    if "lift" in usable_rules_df.columns:
        usable_rules_df = usable_rules_df[usable_rules_df["lift"] > 1].copy()
    if "confidence" in usable_rules_df.columns:
        usable_rules_df = usable_rules_df[usable_rules_df["confidence"] >= 0.05].copy()

    if "rule_str" in usable_rules_df.columns:
        sort_cols = [c for c in ["lift", "confidence", "support", "support_count"] if c in usable_rules_df.columns]
        usable_rules_df = usable_rules_df.sort_values(sort_cols, ascending=[False] * len(sort_cols))

    fpgrowth_rules_preview_df = usable_rules_df.head(10).copy()

    fpgrowth_payload = {
        "rule_source": fpgrowth_rule_source,
        "itemset_source": fpgrowth_itemset_source,
        "n_rules_total_source": int(len(rules_source_df)),
        "n_rules_usable_for_ui": int(len(usable_rules_df)),
        "top_5_rules": fpgrowth_rules_preview_df.head(5).to_dict(orient="records")
    }
    ui_payload_preview["fpgrowth"] = make_json_serializable(fpgrowth_payload)

    add_check("fpgrowth", "rules_nonempty", len(rules_source_df) > 0, notes=f"source={fpgrowth_rule_source}, n_rules_total={len(rules_source_df)}")
    add_check("fpgrowth", "rules_usable_for_ui", len(usable_rules_df) > 0, notes=f"source={fpgrowth_rule_source}, n_rules_usable_for_ui={len(usable_rules_df)}")
    add_check(
        "fpgrowth",
        "rules_preview_recovered_from_fallback",
        fpgrowth_rule_source == "association_rules_csv" or "fallback" in fpgrowth_rule_source,
        notes=f"rule_source={fpgrowth_rule_source}; itemset_source={fpgrowth_itemset_source}",
        severity="support"
    )
else:
    add_check("fpgrowth", "rules_nonempty", False, notes=f"association_rules unavailable | main={note_missing_vs_empty(association_rules_df, ar_err, 'association_rules_not_found')}")
    add_check("fpgrowth", "rules_usable_for_ui", False, notes="Không có rules usable")
    add_check(
        "fpgrowth",
        "rules_preview_recovered_from_fallback",
        False,
        notes="Không có source nào đủ để tạo rules preview",
        severity="support"
    )

display(Markdown("### Frequent itemsets preview"))
display(fpgrowth_itemsets_preview_df if len(fpgrowth_itemsets_preview_df) > 0 else pd.DataFrame({"message": ["Không có frequent itemsets preview."]}))

display(Markdown("### Association rules preview"))
display(fpgrowth_rules_preview_df if len(fpgrowth_rules_preview_df) > 0 else pd.DataFrame({"message": ["Không có association rules usable."]}))

display(Markdown("### FP-Growth preview source"))
display(pd.DataFrame([{
    "rule_source": fpgrowth_rule_source,
    "itemset_source": fpgrowth_itemset_source,
    "main_rules_status": note_missing_vs_empty(association_rules_df, ar_err, "association_rules_not_found"),
    "main_itemsets_status": note_missing_vs_empty(frequent_itemsets_df, fi_err, "frequent_itemsets_not_found"),
    "grid_search_status": note_missing_vs_empty(fpgrowth_grid_df, fg_grid_err, "threshold_search_not_found"),
}]))


### Frequent itemsets preview

,support,itemset_size,itemsets_str,support_count
0,0.279391,1,furniture_decor,202
1,0.272476,1,bed_bath_table,197
2,0.141079,1,housewares,102
3,0.128631,1,baby,93
4,0.098202,1,garden_tools,71
5,0.096819,2,"bed_bath_table, furniture_decor",70
6,0.095436,1,health_beauty,69
7,0.092669,1,sports_leisure,67
8,0.091286,1,cool_stuff,66
9,0.069156,1,computers_accessories,50


### Association rules preview

,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len,consequent_len,antecedents_str,consequents_str,rule_str,support_count
62,0.002766,0.013831,0.002766,1.000000,72.300000,1.0,0.002728,NaN,0.988904,0.200000,1.000000,0.600000,1,1,fashion_childrens_clothes,fashion_bags_accessories,fashion_childrens_clothes → fashion_bags_acces...,2
63,0.013831,0.002766,0.002766,0.200000,72.300000,1.0,0.002728,1.246542,1.000000,0.200000,0.197781,0.600000,1,1,fashion_bags_accessories,fashion_childrens_clothes,fashion_bags_accessories → fashion_childrens_c...,2
65,0.006916,0.016598,0.002766,0.400000,24.100000,1.0,0.002651,1.639004,0.965181,0.133333,0.389873,0.283333,1,1,books_general_interest,market_place,books_general_interest → market_place,2
64,0.016598,0.006916,0.002766,0.166667,24.100000,1.0,0.002651,1.191701,0.974684,0.133333,0.160864,0.283333,1,1,market_place,books_general_interest,market_place → books_general_interest,2
27,0.008299,0.055325,0.008299,1.000000,18.075000,1.0,0.007840,NaN,0.952580,0.150000,1.000000,0.575000,1,1,audio,watches_gifts,audio → watches_gifts,6
28,0.055325,0.008299,0.008299,0.150000,18.075000,1.0,0.007840,1.166707,1.000000,0.150000,0.142887,0.575000,1,1,watches_gifts,audio,watches_gifts → audio,6
51,0.020747,0.020747,0.004149,0.200000,9.640000,1.0,0.003719,1.224066,0.915254,0.111111,0.183051,0.200000,1,1,office_furniture,furniture_living_room,office_furniture → furniture_living_room,3
52,0.020747,0.020747,0.004149,0.200000,9.640000,1.0,0.003719,1.224066,0.915254,0.111111,0.183051,0.200000,1,1,furniture_living_room,office_furniture,furniture_living_room → office_furniture,3
66,0.009682,0.035961,0.002766,0.285714,7.945055,1.0,0.002418,1.349654,0.882682,0.064516,0.259069,0.181319,1,1,construction_tools_safety,home_construction,construction_tools_safety → home_construction,2
67,0.035961,0.009682,0.002766,0.076923,7.945055,1.0,0.002418,1.072845,0.906743,0.064516,0.067899,0.181319,1,1,home_construction,construction_tools_safety,home_construction → construction_tools_safety,2


### FP-Growth preview source

,rule_source,itemset_source,main_rules_status,main_itemsets_status,grid_search_status
0,association_rules_csv,frequent_itemsets_csv,loaded,loaded,loaded


## Cell 15: Tạo UI-ready payload preview

**Mục tiêu**
- Gom tất cả payload inference đã dựng được vào một JSON duy nhất để Streamlit / API layer có thể dùng thử ngay.

**Payload kỳ vọng**
- classification tabular
- classification text
- regression
- clustering
- recommendation (known / unknown user, known / unknown product)
- fpgrowth

**Đầu ra mong đợi**
- `integration_ui_payload_preview.json`
- `integration_ui_payload_index.csv`
- bảng index các payload đang có / chưa có


In [15]:
# =========================
# UI-READY PAYLOAD PREVIEW
# =========================
ui_payload_path = PRED_DIR / "integration_ui_payload_preview.json"
with open(ui_payload_path, "w", encoding="utf-8") as f:
    json.dump(make_json_serializable(ui_payload_preview), f, ensure_ascii=False, indent=2)

print("Đã lưu:", ui_payload_path)

display(Markdown("### Các payload đã sẵn sàng cho Streamlit"))
payload_index_df = pd.DataFrame([
    {"payload_name": k, "present": True, "type": type(v).__name__}
    for k, v in ui_payload_preview.items()
]).sort_values("payload_name").reset_index(drop=True)

payload_index_output_path = METRIC_DIR / "integration_ui_payload_index.csv"
payload_index_df.to_csv(payload_index_output_path, index=False)

print("Đã lưu:", payload_index_output_path)
display(payload_index_df if len(payload_index_df) > 0 else pd.DataFrame({"message": ["Chưa có payload nào được tạo."]}))


Đã lưu: C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project\artifacts\predictions\integration_ui_payload_preview.json


### Các payload đã sẵn sàng cho Streamlit

Đã lưu: C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project\artifacts\metrics\integration_ui_payload_index.csv


,payload_name,present,type
0,classification_tabular,True,dict
1,classification_tabular_unknown_category,True,dict
2,classification_text,True,list
3,clustering,True,dict
4,clustering_edge_case,True,dict
5,fpgrowth,True,dict
6,recommendation_known_product_neighbors,True,dict
7,recommendation_known_user,True,dict
8,recommendation_unknown_product_neighbors,True,dict
9,recommendation_unknown_user,True,dict


## Cell 16: Tổng hợp pass/fail cuối cùng của integration check

**Mục tiêu**
- Tập trung toàn bộ `check_rows` thành 1 bảng summary cuối.
- Quy đổi notebook này thành trạng thái:
  - `PASS`
  - hoặc `FAIL`

**Quy tắc đánh giá**
- Nếu `n_core_failed_checks = 0` → PASS ở mức core
- Nếu còn bất kỳ lỗi core nào → FAIL

**Đầu ra mong đợi**
- `integration_check_summary.csv`
- `integration_check_summary.json`
- bảng failed checks để biết cần sửa upstream artifact ở đâu trước

**Lưu ý**
- Notebook này phải trung thực: không che lỗi, không ép PASS giả.


In [16]:
# guard nếu cell object audit trước đó chưa tạo được path
if "object_audit_output_path" not in globals():
    object_audit_output_path = METRIC_DIR / "integration_object_audit.csv"
if "dependency_contract_output_path" not in globals():
    dependency_contract_output_path = METRIC_DIR / "integration_dependency_contract.csv"
if "payload_index_output_path" not in globals():
    payload_index_output_path = METRIC_DIR / "integration_ui_payload_index.csv"

# =========================
# FINAL PASS / FAIL SUMMARY
# =========================
summary_df = pd.DataFrame(check_rows)
if len(summary_df) == 0:
    summary_df = pd.DataFrame(columns=["module", "check_name", "passed", "severity", "notes"])

summary_df["passed"] = summary_df["passed"].astype(bool)
summary_df = summary_df.sort_values(["passed", "module", "check_name"], ascending=[True, True, True]).reset_index(drop=True)

core_failures = summary_df[(summary_df["severity"].eq("core")) & (~summary_df["passed"])]
all_failures = summary_df[~summary_df["passed"]]

overall_status = "PASS" if len(core_failures) == 0 else "FAIL"

final_summary_payload = {
    "integration_notebook": INTEGRATION_NOTEBOOK_NAME,
    "upstream_notebooks": UPSTREAM_NOTEBOOKS,
    "optional_non_dependency_notebooks": OPTIONAL_NON_DEPENDENCY_NOTEBOOKS,
    "overall_status": overall_status,
    "n_total_checks": int(len(summary_df)),
    "n_passed_checks": int(summary_df["passed"].sum()),
    "n_failed_checks": int((~summary_df["passed"]).sum()),
    "n_core_failed_checks": int(len(core_failures)),
    "core_failed_checks": core_failures[["module", "check_name", "notes"]].to_dict(orient="records"),
    "all_failed_checks": all_failures[["module", "check_name", "notes"]].to_dict(orient="records"),
    "saved_outputs": {
        "file_registry": str(file_registry_output_path),
        "dependency_contract": str(dependency_contract_output_path),
        "object_audit": str(object_audit_output_path),
        "schema_audit": str(schema_audit_output_path),
        "ui_payload_preview": str(ui_payload_path),
        "ui_payload_index": str(payload_index_output_path)
    }
}

summary_csv_path = METRIC_DIR / "integration_check_summary.csv"
summary_json_path = METRIC_DIR / "integration_check_summary.json"

summary_df.to_csv(summary_csv_path, index=False)
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(make_json_serializable(final_summary_payload), f, ensure_ascii=False, indent=2)

print("Đã lưu:", summary_csv_path)
print("Đã lưu:", summary_json_path)

display(Markdown("### Final integration summary"))
display(pd.DataFrame([{
    "overall_status": final_summary_payload["overall_status"],
    "n_total_checks": final_summary_payload["n_total_checks"],
    "n_passed_checks": final_summary_payload["n_passed_checks"],
    "n_failed_checks": final_summary_payload["n_failed_checks"],
    "n_core_failed_checks": final_summary_payload["n_core_failed_checks"],
}]))

display(Markdown("### Failed checks"))
display(all_failures if len(all_failures) > 0 else pd.DataFrame({"message": ["Không có failed checks. Integration đã pass ở mức core."]}))


Đã lưu: C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project\artifacts\metrics\integration_check_summary.csv
Đã lưu: C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project\artifacts\metrics\integration_check_summary.json


### Final integration summary

,overall_status,n_total_checks,n_passed_checks,n_failed_checks,n_core_failed_checks
0,PASS,43,43,0,0


### Failed checks

,message
0,Không có failed checks. Integration đã pass ở ...


## Kết luận cách đọc notebook này

### Khi nào được xem là PASS
- `overall_status = PASS`
- `n_core_failed_checks = 0`

Khi đó có thể hiểu rằng:
- artifact lõi đã đủ,
- model/bundle chính load được,
- inference smoke test của các nhánh chính chạy được,
- payload tối thiểu cho UI đã sẵn sàng.

### Khi nào phải xem là FAIL
Nếu còn fail:
- đọc `integration_check_summary.csv`
- ưu tiên các dòng có `severity = core`
- sửa notebook upstream tương ứng rồi **chạy lại notebook 08**

### File đầu ra do notebook 08 ghi thêm
Notebook này **chỉ ghi các file audit/integration riêng**, không sửa file upstream:
- `artifacts/metrics/integration_file_registry.csv`
- `artifacts/metrics/integration_dependency_contract.csv`
- `artifacts/metrics/integration_object_audit.csv`
- `artifacts/metrics/integration_schema_audit.csv`
- `artifacts/metrics/integration_ui_payload_index.csv`
- `artifacts/metrics/integration_check_summary.csv`
- `artifacts/metrics/integration_check_summary.json`
- `artifacts/predictions/integration_ui_payload_preview.json`

### File upstream được đọc nhưng không bị sửa
- Từ notebook 02:
  - `orders_base_final.*`
  - `rfm_df.*`
  - `ratings_df.*`
  - `transactions_df.*`
  - `transactions_order_category_df.*`
  - `product_lookup.*`
- Từ notebook 03:
  - classification summary + classifier artifacts
- Từ notebook 04:
  - regression summary + regressor artifact
- Từ notebook 05:
  - clustering summary + scaler/PCA/model/profile artifacts
- Từ notebook 06:
  - recommendation summary + Surprise artifacts + helper CSV
- Từ notebook 07:
  - FP-Growth summary + itemsets/rules/preview artifacts

### File upstream **không nằm trong dependency bắt buộc của notebook 08**
- `01_eda_data_structure_check.ipynb`
- `09_chi_square_quick_check.ipynb`

### Ảnh hưởng và không ảnh hưởng
- **Có ảnh hưởng logic**: notebook 02 → 07 vì notebook 08 đọc output của chúng
- **Không bị notebook 08 sửa đổi**: toàn bộ notebook 02 → 07
- **Không bị ảnh hưởng downstream trực tiếp trong code**: notebook 01 và 09

> Tóm lại: notebook 08 là lớp kiểm tra tích hợp cuối, dùng để xác nhận pipeline đã đủ sẵn sàng nối vào Streamlit ở mức inference, chứ không phải để train lại hoặc thay đổi artifact upstream.


## CHECK-IN — FILE 08: `08_integration_check.ipynb`

### A. THIẾT LẬP BAN ĐẦU
- [x] Hoàn thành import thư viện cần thiết cho integration check
- [x] Hoàn thành khai báo tên notebook integration
- [x] Hoàn thành khai báo danh sách notebook upstream:
  - `02_data_preprocessing_master_dataset.ipynb`
  - `03_classification_pipeline.ipynb`
  - `04_regression_pipeline.ipynb`
  - `05_clustering_rfm.ipynb`
  - `06_surprise_recommendation.ipynb`
  - `07_fpgrowth_rules.ipynb`
- [x] Hoàn thành khai báo danh sách notebook non-dependency tùy chọn
- [x] Hoàn thành khai báo cấu hình thư mục:
  - `BASE_DIR`
  - `PROCESSED_DIR`
  - `ARTIFACT_DIR`
  - `MODEL_DIR`
  - `METRIC_DIR`
  - `PLOT_DIR`
  - `PRED_DIR`

---

### B. ĐỊNH VỊ PROJECT VÀ CHUẨN HÓA PATH
- [x] Hoàn thành locate project base
- [x] Hoàn thành resolve thư mục processed
- [x] Hoàn thành resolve thư mục artifacts
- [x] Hoàn thành bảo đảm thư mục metric output tồn tại
- [x] Hoàn thành hỗ trợ fallback path cho nhiều môi trường chạy notebook

---

### C. HÀM HELPER CHO INTEGRATION CHECK
- [x] Hoàn thành định nghĩa helper audit:
  - `add_check`
  - `add_object_audit`
  - `add_schema_audit`
- [x] Hoàn thành định nghĩa helper path / IO:
  - `resolve_saved_path`
  - `safe_read_json`
  - `safe_joblib_load`
  - `safe_pickle_load`
  - `safe_read_table`
- [x] Hoàn thành định nghĩa helper utility / serialization:
  - `unique_preserve_order`
  - `normalize_text`
  - `make_json_serializable`
  - `first_non_null_value`
- [x] Hoàn thành định nghĩa helper extract / infer schema:
  - `try_get_pipeline_from_estimator`
  - `flatten_transformer_columns`
  - `extract_feature_columns_from_estimator`
  - `infer_categorical_columns_from_sample`

---

### D. FILE REGISTRY VÀ SUMMARY JSON
- [x] Hoàn thành khai báo `registry` mặc định cho artifact integration
- [x] Hoàn thành khai báo nhóm file trong registry:
  - processed tables
  - summary JSON
  - model / bundle
  - recommendation helper files
  - clustering profile files
  - FP-Growth output files
- [x] Hoàn thành đọc:
  - `classification_summary`
  - `regression_summary`
  - `clustering_summary`
  - `surprise_summary`
  - `fpgrowth_summary`
- [x] Hoàn thành tạo bảng trạng thái load summary JSON
- [x] Hoàn thành cập nhật registry từ summary JSON upstream
- [x] Hoàn thành tạo `file_registry_df`
- [x] Hoàn thành lưu:
  - `integration_file_registry.csv`
  - `integration_dependency_contract.csv`

---

### E. KIỂM TRA FILE ARTIFACT CỐT LÕI
- [x] Hoàn thành audit tồn tại các file artifact cốt lõi
- [x] Hoàn thành tạo check `core_required_files_exist`
- [x] Hoàn thành tổng hợp dependency contract của integration

---

### F. LOAD CÁC BẢNG PROCESSED
- [x] Hoàn thành load `orders_base_final`
- [x] Hoàn thành load `rfm_df`
- [x] Hoàn thành load `ratings_df`
- [x] Hoàn thành load `transactions_df`
- [x] Hoàn thành load `transactions_order_category_df`
- [x] Hoàn thành load `product_lookup_processed`
- [x] Hoàn thành tạo bảng summary load processed tables
- [x] Hoàn thành kiểm tra tải dữ liệu tối thiểu cho integration

---

### G. LOAD MODEL / BUNDLE / ARTIFACT TỪ NOTEBOOK UPSTREAM
- [x] Hoàn thành load artifact classification:
  - model tabular
  - model text
- [x] Hoàn thành load artifact regression:
  - regressor model
- [x] Hoàn thành load artifact clustering:
  - `kmeans_model`
  - `gmm_model`
  - `rfm_scaler`
  - `rfm_pca`
- [x] Hoàn thành load artifact recommendation:
  - `best_surprise_model`
  - `seen_items_map`
  - `surprise_deployment_bundle`
  - `item_knn_neighbors_model`
  - helper tables known users / items / candidates / fallback
- [x] Hoàn thành load artifact FP-Growth:
  - `frequent_itemsets.csv`
  - `association_rules.csv`
  - `fpgrowth_threshold_search.csv`
  - preview / report files
- [x] Hoàn thành tạo bảng object audit
- [x] Hoàn thành lưu `integration_object_audit.csv`

---

### H. TRÍCH XUẤT SCHEMA ĐẦU VÀO CHO TỪNG MODULE
- [x] Hoàn thành trích xuất schema classification
- [x] Hoàn thành trích xuất schema regression
- [x] Hoàn thành trích xuất schema clustering
- [x] Hoàn thành fallback schema từ:
  - `preprocessor.transformers_`
  - `transformers`
  - `feature_names_in_`
  - summary JSON
  - feature list mặc định
- [x] Hoàn thành tạo `schema_audit_df`
- [x] Hoàn thành lưu `integration_schema_audit.csv`

---

### I. SMOKE TEST CHO CLASSIFICATION
- [x] Hoàn thành tạo sample inference tabular từ `orders_base_final`
- [x] Hoàn thành fill tối thiểu cho sample classification
- [x] Hoàn thành chạy `.predict()` cho classifier tabular
- [x] Hoàn thành chạy `.predict_proba()` hoặc `decision_function()` nếu có
- [x] Hoàn thành tạo `classification_payload`
- [x] Hoàn thành tạo `classification_edge_payload`
- [x] Hoàn thành tạo payload text classification
- [x] Hoàn thành kiểm tra text inference với:
  - text tích cực
  - text tiêu cực
  - chuỗi rỗng
- [x] Hoàn thành thêm payload classification vào `ui_payload_preview`
- [x] Hoàn thành tạo check cho classification smoke test

---

### J. SMOKE TEST CHO REGRESSION
- [x] Hoàn thành tạo sample inference regression từ `orders_base_final`
- [x] Hoàn thành chạy `.predict()` cho regressor model
- [x] Hoàn thành tạo `regression_payload`
- [x] Hoàn thành thêm:
  - `y_true_reference`
  - `absolute_error_vs_reference`
- [x] Hoàn thành tạo `regression_edge_payload`
- [x] Hoàn thành kiểm tra unknown categorical values cho regression
- [x] Hoàn thành thêm payload regression vào `ui_payload_preview`
- [x] Hoàn thành tạo check cho regression smoke test

---

### K. CHỌN MÔ HÌNH CLUSTERING ĐỂ SUY LUẬN
- [x] Hoàn thành logic chọn model clustering inference
- [x] Hoàn thành hỗ trợ chọn giữa:
  - `kmeans`
  - `gmm`
- [x] Hoàn thành đọc:
  - `kmeans_profile_df`
  - `gmm_profile_df`
- [x] Hoàn thành map label cột:
  - `cluster_kmeans`
  - `cluster_gmm`
- [x] Hoàn thành tạo helper map cluster → segment

---

### L. SMOKE TEST CHO CLUSTERING
- [x] Hoàn thành tạo sample clustering từ `rfm_df`
- [x] Hoàn thành scale sample bằng `rfm_scaler`
- [x] Hoàn thành chạy suy luận bằng model clustering đã chọn
- [x] Hoàn thành map cluster sang `segment_name`
- [x] Hoàn thành tạo `clustering_payload`
- [x] Hoàn thành tạo `clustering_edge_payload`
- [x] Hoàn thành thêm payload clustering vào `ui_payload_preview`
- [x] Hoàn thành tạo check cho clustering inference

---

### M. HELPER CHO RECOMMENDATION
- [x] Hoàn thành định nghĩa helper chuẩn hóa `product_lookup`
- [x] Hoàn thành định nghĩa helper gộp cột metadata lookup
- [x] Hoàn thành định nghĩa helper enrich metadata sản phẩm
- [x] Hoàn thành định nghĩa helper build fallback output
- [x] Hoàn thành định nghĩa helper recommend top-N cho user
- [x] Hoàn thành định nghĩa helper recommend neighbor theo `product_id`

---

### N. SMOKE TEST CHO RECOMMENDATION
- [x] Hoàn thành chuẩn bị:
  - `known_users_set`
  - `candidate_item_ids`
  - `top_n_default`
- [x] Hoàn thành smoke test cho:
  - known user
  - unknown user fallback
  - known product neighbors
  - unknown product fallback
- [x] Hoàn thành tạo:
  - `recommendation_payload_known_user`
  - `recommendation_payload_unknown_user`
  - `neighbor_payload_known_item`
  - `neighbor_payload_unknown_item`
- [x] Hoàn thành thêm payload recommendation vào `ui_payload_preview`
- [x] Hoàn thành tạo check cho:
  - `known_user_top_n`
  - `unknown_user_fallback`
  - `known_product_neighbors`
  - `unknown_product_neighbors_fallback`

---

### O. SMOKE TEST CHO FP-GROWTH
- [x] Hoàn thành schema audit cho:
  - `frequent_itemsets`
  - `association_rules`
  - preview fallback files
- [x] Hoàn thành xây dựng preview itemsets từ:
  - source chính
  - fallback preview
- [x] Hoàn thành xây dựng preview rules từ:
  - source chính
  - fallback report
  - fallback preview
- [x] Hoàn thành chuẩn hóa và sắp xếp:
  - `support`
  - `support_count`
  - `confidence`
  - `lift`
- [x] Hoàn thành tạo:
  - `fpgrowth_itemsets_preview_df`
  - `fpgrowth_rules_preview_df`
  - `fpgrowth_payload`
- [x] Hoàn thành thêm payload FP-Growth vào `ui_payload_preview`
- [x] Hoàn thành tạo check cho artifact lõi và preview của FP-Growth

---

### P. TẠO UI-READY PAYLOAD PREVIEW
- [x] Hoàn thành ghi `integration_ui_payload_preview.json`
- [x] Hoàn thành tạo `payload_index_df`
- [x] Hoàn thành lưu `integration_ui_payload_index.csv`
- [x] Hoàn thành tổng hợp các payload đã sẵn sàng cho Streamlit

---

### Q. TỔNG HỢP PASS / FAIL CUỐI CÙNG
- [x] Hoàn thành tạo `summary_df` từ toàn bộ `check_rows`
- [x] Hoàn thành phân loại:
  - passed checks
  - failed checks
  - core failures
- [x] Hoàn thành tính:
  - `overall_status`
  - `n_total_checks`
  - `n_passed_checks`
  - `n_failed_checks`
  - `n_core_failed_checks`
- [x] Hoàn thành tạo `final_summary_payload`
- [x] Hoàn thành lưu:
  - `integration_check_summary.csv`
  - `integration_check_summary.json`

---

### R. OUTPUT FILE DO NOTEBOOK 08 TẠO RA
- [x] Hoàn thành tạo:
  - `integration_file_registry.csv`
  - `integration_dependency_contract.csv`
  - `integration_object_audit.csv`
  - `integration_schema_audit.csv`
  - `integration_ui_payload_index.csv`
  - `integration_check_summary.csv`
  - `integration_check_summary.json`
  - `integration_ui_payload_preview.json`

---

### S. KẾT QUẢ ĐẦU RA CHÍNH CỦA NOTEBOOK
- [x] Hoàn thành phần **registry mặc định và update registry từ summary JSON**
- [x] Hoàn thành phần **kiểm tra file artifact cốt lõi**
- [x] Hoàn thành phần **load processed tables**
- [x] Hoàn thành phần **load model / bundle / artifact từ notebook upstream**
- [x] Hoàn thành phần **schema audit cho classification / regression / clustering**
- [x] Hoàn thành phần **classification smoke test**
- [x] Hoàn thành phần **regression smoke test**
- [x] Hoàn thành phần **clustering inference test**
- [x] Hoàn thành phần **recommendation helper và recommendation smoke test**
- [x] Hoàn thành phần **FP-Growth smoke test**
- [x] Hoàn thành phần **UI-ready payload preview**
- [x] Hoàn thành phần **final pass/fail summary**
- [x] Hoàn thành phần **ghi các file audit / integration output**
- [x] Hoàn thành phần **kết luận cách đọc notebook integration**

### Kết luận cuối:
Notebook `08_integration_check.ipynb` đã **hoàn thành** toàn bộ các hạng mục chính thuộc phạm vi **Artifact Registry, Dependency Audit, Object Audit, Schema Audit, Classification/Regression/Clustering/Recommendation/FP-Growth Smoke Tests, UI Payload Preview, Integration Summary, và Integration Output Files** của riêng file này.